# **Dự Đoán Chất Lượng Không Khí Cho Ngày Hôm Sau Ở TP.HCM**

Chất lượng không khí tại TP. Hồ Chí Minh ngày càng trở thành mối quan tâm lớn của người dân, đặc biệt với chỉ số AQI trung bình năm 2022-2026 đạt 81.6 (mức Trung bình theo thang US), trong đó nhiều ngày vượt ngưỡng 100 - mức Không tốt cho nhóm nhạy cảm. Khả năng dự báo AQI trước một ngày giúp người dân lên kế hoạch sinh hoạt, hạn chế phơi nhiễm, và hỗ trợ các cơ quan quản lý môi trường.

### **Mục tiêu notebook này:**

- Xây dựng và huấn luyện các mô hình Machine Learning để dự đoán giá trị US AQI ngày hôm sau (bài toán Regression).
- So sánh hiệu suất giữa các mô hình: Ridge Regression, Decision Tree, Random Forest, XGBoost, LightGBM, và SARIMA (mô hình thống kê cổ điển để đối chứng).
- Kiểm tra độ ổn định của từng mô hình bằng Cross-Validation theo thời gian (TimeSeriesSplit), không chỉ dựa vào kết quả Test set đơn.
- Phát hiện và xử lý hiện tượng overfitting (so sánh R² Train vs Test).
- Phân tích Feature Importance để hiểu mô hình dựa vào yếu tố nào để dự đoán.
- Chứng minh giá trị của Feature Engineering bằng cách so sánh ML với SARIMA.
- Lựa chọn Best Model dựa trên cả độ chính xác (R², RMSE, MAE, MAPE) và độ ổn định (CV Mean ± Std).
- Lưu Best Model (`best_regressor.pkl`) và metadata (`metadata.json`) để sử dụng ở SHAP và trong Dashboard.

## **00. Import và cấu hình**

In [ ]:
# Standard Libraries
import warnings
import gdown
import os
import pickle
import joblib
import json
import copy
from google.colab import drive
from datetime import datetime

# Data Manipulation - Math
import numpy as np
import pandas as pd

# Data Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import folium
from matplotlib.colors import LinearSegmentedColormap
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy.stats import gaussian_kde
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.colorbar import ColorbarBase

# Machine Learning - Preprocessing
import lightgbm as lgb
import xgboost as xgb
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX

# !pip install pmdarima
from pmdarima import auto_arima
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
)
from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import RobustScaler

In [ ]:
# Cấu hình
warnings.filterwarnings(
    "ignore"
)  # Tắt các cảnh báo không cần thiết cho notebook sạch hơn
pd.set_option(
    "display.float_format", "{:.2f}".format
)  # Hiển thị số thập phân gọn hơn (làm tròn 2 chữ số thập phân)
plt.rcParams.update(
    {
        "figure.dpi": 150,
        "axes.grid": True,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.family": "DejaVu Sans",
    }
)  # Cấu hình style mặc định cho toàn bộ biểu đồ matplotlib trong notebook

# AQI Color Palette
# Đây là màu chính thức của US EPA, để đảm bảo nhất quán toàn bộ notebook
AQI_COLORS = {
    "Good": "#00E400",
    "Moderate": "#FFFF00",
    "Unhealthy for Sensitive Groups": "#FF7E00",
    "Unhealthy": "#FF0000",
    "Very Unhealthy": "#8F3F97",
    "Hazardous": "#7E0023",
}

AQI_BINS = [0, 50, 100, 150, 200, 300, 500]
AQI_LABELS = list(AQI_COLORS.keys())
AQI_MAPPING = {label: idx for idx, label in enumerate(AQI_LABELS)}


# Hàm AQI_CATEGORY()
def AQI_CATEGORY(value):
    """
    Phân loại mức AQI theo thang US EPA
    Input : Giá trị AQI (số thực)
    Output: Tên mức (string)
    """
    if value <= 50:
        return "Good"
    elif value <= 100:
        return "Moderate"
    elif value <= 150:
        return "Unhealthy for Sensitive Groups"
    elif value <= 200:
        return "Unhealthy"
    elif value <= 300:
        return "Very Unhealthy"
    else:
        return "Hazardous"


# Tạo Custom Gradient Colormap từ AQI_COLORS và AQI_BINS
# Chuẩn hóa các mốc AQI vì LinearSegmentedColormap là thang đo 0.0 đến 1.0
MAX_AQI = 500
positions = [
    val / MAX_AQI for val in AQI_BINS
]  # positions sẽ trở thành [0.0, 0.1, 0.2, 0.3, 0.4, 0.6, 1.0]

# Vì AQI_BINS có 7 mốc nhưng AQI_COLORS chỉ có 6 màu
colors = list(AQI_COLORS.values())
gradient_colors = colors + [colors[-1]]  # Nhân đôi màu cuối cùng

# Ghép vị trí và màu sắc lại để tạo dải gradient
color_mapping = list(
    zip(positions, gradient_colors)
)  # Gán các màu tương ứng với từng giá trị
AQI_gradient_cmap = LinearSegmentedColormap.from_list(
    "AQI_gradient", color_mapping
)  # Tạo ra dãy màu liên tục thay vì riêng lẻ

In [ ]:
# drive.mount('/content/drive')
ROOT = "/content/drive/MyDrive/HCMUS/Nhập Môn Khoa Học Dữ Liệu/Mini Project"

figures_dir = os.path.join(ROOT, "figures")
models_dir = os.path.join(figures_dir, "models")

os.makedirs(figures_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)

In [ ]:
folder_ID = "1b8LGMtOwMrj6FDMODA8-rJNq0cKXlgBA"
folder_url = f"https://drive.google.com/drive/folders/{folder_ID}"

gdown.download_folder(folder_url, output="data", quiet=False)

X_train = pd.read_csv("data/X_train.csv", index_col="date", parse_dates=True)
X_test = pd.read_csv("data/X_test.csv", index_col="date", parse_dates=True)
y_train = pd.read_csv("data/y_train.csv", index_col="date", parse_dates=True)
y_test = pd.read_csv("data/y_test.csv", index_col="date", parse_dates=True)
df_processed = pd.read_csv(
    "data/data_processed.csv", index_col="date", parse_dates=True
)

y_train_reg = y_train["target_reg_tomorrow"]
y_test_reg = y_test["target_reg_tomorrow"]

## **03. Model Regression**

In [ ]:
# Hàm đánh giá
def evaluate(name, y_true, y_pred):
    y_pred = np.clip(y_pred, 0, 500)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100

    print(f"{name} - Kết quả trên tập Test:")
    print(f"   MAE  = {mae:.2f}    (Sai số tuyệt đối trung bình)")
    print(f"   RMSE = {rmse:.2f}   (Căn bậc hai sai số bình phương trung bình)")
    print(f"   MSE  = {mse:.2f}  (Sai số bình phương trung bình)")
    print(f"   R²   = {r2:.4f}  (Hệ số xác định - càng gần 1 càng tốt)")
    print(f"   MAPE = {mape:.2f}%   (Sai số phần trăm trung bình)")

In [ ]:
def cv_evaluate(name, model_class, model_params, X, y, n_splits=5, lo=0, hi=500):
    """
    Cross-validation theo thời gian
    """
    tscv = TimeSeriesSplit(n_splits=n_splits)
    fold_results = []

    for fold, (tr_idx, te_idx) in enumerate(tscv.split(X), 1):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

        model = model_class(**model_params)
        model.fit(X_tr, y_tr)

        pred_raw = model.predict(X_te)
        pred_clipped = np.clip(pred_raw, lo, hi)

        r2 = r2_score(y_te, pred_clipped)
        mae = mean_absolute_error(y_te, pred_clipped)
        rmse = np.sqrt(mean_squared_error(y_te, pred_clipped))

        fold_results.append(
            {
                "Model": name,
                "Fold": fold,
                "n_train": len(tr_idx),
                "n_test": len(te_idx),
                "R²": round(r2, 4),
                "MAE": round(mae, 2),
                "RMSE": round(rmse, 2),
                "Pred_min_raw": round(pred_raw.min(), 1),
                "Pred_max_raw": round(pred_raw.max(), 1),
            }
        )

    return pd.DataFrame(fold_results)


def cv_summary(df_folds):
    """Tổng hợp Mean ± Std cho mỗi model"""
    return (
        df_folds.groupby("Model")
        .agg(
            R2_mean=("R²", "mean"),
            R2_std=("R²", "std"),
            MAE_mean=("MAE", "mean"),
            RMSE_mean=("RMSE", "mean"),
        )
        .round(4)
        .reset_index()
    )

### **3.1. Ridge Regression (Baseline)**

Ridge Regression là mô hình hồi quy tuyến tính có thêm L2 Regularization.

Mô hình này được dùng làm **Baseline** - đơn giản, huấn luyện nhanh giúp đánh giá xem các mô hình phức tạp hơn có thực sự cải thiện không.

In [ ]:
# Định nghĩa mô hình Ridge Regression
ridge_reg = Ridge(
    alpha=10.0,  # Hệ số L2 regularization - càng lớn càng tránh overfit
    fit_intercept=True,  # Có tính hệ số chặn hay không
    max_iter=1000,  # Số vòng lặp tối đa
    random_state=42,
)

# Train mô hình
ridge_reg.fit(X_train, y_train_reg)

#### **Nhận xét Ridge Regression:**

- **Baseline model:** Mô hình tuyến tính đơn giản nhất, dùng làm mốc so sánh với các mô hình phức tạp hơn như XGBoost, LightGBM.

- **L2 Regularization (alpha = 10.0):** Phạt các hệ số lớn → giảm overfitting so với Linear Regression thông thường (có thể thử nhiều alpha và chọn ra cái ổn định nhất).

- **Giới hạn:** Chỉ học được mối quan hệ tuyến tính → có thể không đủ mạnh cho dữ liệu chất lượng không khí có nhiều pattern phi tuyến theo mùa vụ.

- **Ridge có coef_** - có thể xem hệ số nào lớn nhất để biết feature nào ảnh hưởng nhiều.

In [ ]:
# Dự đoán và đánh giá
y_pred_ridge = ridge_reg.predict(X_test)
y_pred_ridge = np.clip(y_pred_ridge, 0, 500)

evaluate("Ridge Regression", y_test_reg, y_pred_ridge)

#### **Nhận xét alpha = 1.0 và alpha = 10.0**
- **Khi alpha = 10,** mô hình bị ép chặt hơn (hệ số nhỏ hơn), khiến nó ổn định và dự đoán khá sát cho phần lớn các điểm dữ liệu bình thường (MAE, MAPE tốt).

- **Ngược lại, alpha = 1** linh hoạt hơn một chút, giúp nó bắt được các điểm dữ liệu khó đoán đó tốt hơn (MSE, RMSE tốt), nhưng lại hi sinh đi một chút xíu độ chính xác tổng thể (MAE, MAPE kém hơn một chút).

- alpha thấp áp dụng cho những bài có toán dự đoán sai lệch ít, chấp nhận sai số nhỏ ở mọi nơi còn hơn là dính một vài cú sai lệch khổng lồ. Còn alpha cao áp dụng cho những bài toán cần độ chính xác vài ngoại lệ sai số lớn không quá quan trọng.

> **Với alpha = 10.0,** mô hình đã đạt được hiệu suất tối ưu. Tại đây, mô hình vừa khắc phục được hiện tượng overfit, giữ được độ tổng quát hóa cao, vừa giảm thiểu đồng thời cả sai số trung bình (MAE = 8.27) lẫn các sai số đột biến (RMSE = 10.78, thấp nhất trong các thử nghiệm), đồng thời đưa hệ số xác định R² lên mức cao nhất (0.7491). Đây là quá trình thử sai với tham số alpha.

#### **Đánh giá các metrics:**

- **R² = 0.7491:** Mô hình giải thích được khoảng 75% sự biến thiên của dữ liệu. Đây là con số ấn tượng với mô hình tuyến tính đơn giản - cho thấy các feature có mối quan hệ tương đối tuyến tính với AQI ngày mai (Trong các bài toán thực tế, mức R² > 0.7 được đánh giá là một mô hình tốt và có độ tin cậy cao).

- **MAPE = 8.86%:** Dự đoán của mô hình chỉ lệch khoảng 9% so với con số thực tế - tương đương độ chính xác ~91%, khá tốt cho một mô hình Baseline (Trong thực tế, một mô hình dự đoán có MAPE dưới 10% (độ chính xác > 90%) thường được coi là xuất sắc và có giá trị cao để ra quyết định).

- **MAE = 8.27:** Trung bình mỗi dự đoán lệch 8.26 đơn vị AQI so với thực tế.

- **Điểm cần lưu ý: RMSE = 10.78 cao hơn MAE** → mô hình có một số dự đoán sai lệch lớn, đặc biệt vào những ngày ô nhiễm nặng mà mô hình tuyến tính khó nắm bắt. Nhưng khoảng cách giữa RMSE và MAE cũng tương đối hẹp (chênh lệch khoảng 2.51), điều này chứng tỏ mô hình rất ổn định và không tạo ra nhiều lỗi dự đoán nghiêm trọng.

> **Tóm lại:** Tại ngưỡng tối ưu alpha = 10, mô hình Ridge Regression đạt độ tin cậy cao với tỷ lệ giải thích dữ liệu (R²) lên tới 74.91%. Sai số dự đoán trung bình (MAPE) được duy trì ở mức thấp, dưới 9%. Đặc biệt, khoảng cách hẹp giữa chỉ số RMSE (10.78) và MAE (8.27) cho thấy mô hình hoạt động vô cùng ổn định, kiểm soát tốt hiện tượng nhiễu và hạn chế tối đa các dự đoán sai lệch lớn.

In [ ]:
# Xem hệ số (coefficients)
coef_ridge = (
    pd.Series(ridge_reg.coef_, index=X_train.columns)
    .sort_values(key=abs, ascending=False)
    .head(20)
)

plt.figure(figsize=(10, 6))
coef_ridge[::-1].plot(kind="barh", color="#3498DB", alpha=0.85)
plt.xlabel("Coefficient Value")
plt.title("Ridge - Top 20 Hệ số (|coef| lớn nhất)")
plt.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
file_path = os.path.join(models_dir, "ridge_coefficients")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()


#### **Kiểm tra Ridge bằng Cross-Validation**

**Mục đích cốt lõi của Cross-Validation (Xác thực chéo)** là để đánh giá chính xác và khách quan nhất khả năng dự đoán của mô hình trên dữ liệu mới (dữ liệu mà nó chưa từng nhìn thấy), từ đó giúp đảm bảo mô hình học được "quy luật thực sự" thay vì chỉ "học vẹt" trên tập dữ liệu huấn luyện.

In [ ]:
df_cv_ridge = cv_evaluate(
    "Ridge Regression", Ridge, {"alpha": 10}, X_train, y_train_reg
)
display(df_cv_ridge)
display(cv_summary(df_cv_ridge))


**Đánh giá:**

- **Ridge (alpha=10):** R² Mean = 0.70 ± 0.10

- **Sau khi tăng alpha từ giá trị mặc định lên 10.0, Ridge đạt kết quả
ổn định:** R² CV Mean = 0.70 ± 0.10, không còn fold nào cho giá trị
âm hoặc dự đoán AQI âm (Pred_min luôn dương qua tất cả 5 folds).

- **So sánh với kết quả Test set đơn (R² = 0.7491):** CV Mean = 0.70 gần
tương đương, cho thấy con số 0.7491 không phải ăn may mà phản
ánh đúng năng lực thực sự của Ridge khi đã lựa chọn alpha hợp lý.

#### **Kết luận mô hình Ridge Regression:**

- **Đánh giá tổng thể:** Mô hình Baseline đạt kết quả tốt hơn kỳ vọng - cho thấy AQI ngày mai có mối quan hệ tuyến tính đáng kể với các feature hiện tại.

- **Kết quả: MAE = 8.27 | RMSE = 10.78 | R² = 0.7491 | MAPE = 8.86%**

**Đánh giá:** R² = 0.7491 cho thấy các feature có mối quan hệ tuyến tính khá tốt với AQI ngày mai - đặc biệt các feature như pm2_5, là feature ảnh hưởng lớn nhất tới mô hình

- **Ý nghĩa thực tiễn:** MAE = 8.27 nghĩa là sai số dự đoán AQI trung bình là ±8.27 đơn vị - chấp nhận được cho bài toán dự đoán môi trường. MAPE = 8.86% cho thấy model dự đoán sai ~9% so với giá trị thực.

- **Giới hạn:** Là mô hình tuyến tính nên không nắm bắt được các pattern phi tuyến - đặc biệt vào những ngày AQI biến động bất thường do thời tiết cực đoan.

- **Hướng cải thiện:** Thử các mô hình khác.

> **Kết luận:** Ridge với alpha = 10 là một ứng viên ĐÁNG TIN CẬY cho Best Model, cần so sánh tiếp với CV của Decision Tree, Random Forest,
LightGBM, XGBoost để đưa ra quyết định cuối cùng.


In [ ]:
# Lưu mô hình
models_dir = os.path.join(ROOT, "models")
os.makedirs(models_dir, exist_ok=True)
save_path = os.path.join(models_dir, "ridge_regressor.pkl")
joblib.dump(ridge_reg, save_path)
print("Đã lưu ridge_regressor.pkl")

### **3.2. Decision Tree**

Decision Tree là mô hình cây quyết định đơn giản nhất - chia nhỏ tập dữ liệu thành các nhóm đồng nhất dựa trên các điều kiện (luật "Nếu - Thì"). Thường dùng làm baseline cho các mô hình ensemble (Random Forest, XGBoost, LightGBM).

Mô hình này có khả năng tự động tìm ra các mối quan hệ phi tuyến tính phức tạp giữa các chỉ số ô nhiễm và chỉ số AQI ngày mai mà không cần giả định dữ liệu tuân theo phân phối nào.

**Lưu ý:** Với dataset chỉ ~1013 mẫu Train, Decision Tree rất dễ overfitting nếu để `max_depth` quá sâu - cần giới hạn chặt.

In [ ]:
# Định nghĩa mô hình Decision
dt_reg = DecisionTreeRegressor(
    max_depth=3,  # Giới hạn thấp - dataset nhỏ
    min_samples_leaf=30,  # Mỗi lá phải có tối thiểu 30 mẫu
    random_state=42,
)

# Train mô hình
dt_reg.fit(X_train, y_train_reg)


**Nhận xét Decision Tree:**

- **Lý do chọn max_depth = 3:** Kiểm soát độ phức tạp và tăng tính diễn giải và tính tổng quát.

- **Lý do chọn min_samples_leaf = 30:** Tránh các quyết định dựa trên thiểu số (Outliers) và tăng độ ổn định cho bài toán.

> **Tóm lại:** Sự kết hợp giữa max_depth = 3 và min_samples_leaf = 30 đóng vai trò như những kỹ thuật Regularization. Chúng chủ động ràng buộc Decision Tree lại, không cho nó mô hình hóa quá chi tiết, nhằm đảm bảo một mô hình đơn giản, ổn định và phù hợp nhất với điều kiện dữ liệu hạn chế.

In [ ]:
# Dự đoán và đánh giá
y_pred_dt = dt_reg.predict(X_test)
y_pred_dt = np.clip(y_pred_dt, 0, 500)

evaluate("Decision Tree Regressor", y_test_reg, y_pred_dt)

#### **Đánh giá các metrics:**

- **R² = 0.5675:** Mô hình chỉ giải thích được khoảng 56.75% sự biến thiên của dữ liệu. Đây là một con số ở mức trung bình. Nó cho thấy cấu trúc hiện tại của Decision Tree (có thể do bị giới hạn độ sâu và số lượng mẫu ở lá) chưa đủ phức tạp để nắm bắt toàn bộ xu hướng của dữ liệu AQI, dẫn đến hiệu suất thấp hơn rõ rệt so với mô hình tuyến tính (đạt ~75%).

- **MAPE = 10.51%:** Dự đoán của mô hình lệch khoảng 10.5% so với thực tế. Độ chính xác tương đương khoảng 89.5% là một mức có thể chấp nhận được, nhưng đã vượt qua ngưỡng ổn định trong thực tế (<10%).

- **MAE = 10.21:** Trung bình mỗi dự đoán lệch 10.21 đơn vị AQI so với thực tế.

- **Điểm cần lưu ý: RMSE = 14.15 cao hơn khá nhiều so với MAE** → Khoảng chênh lệch (gần 4 đơn vị) chỉ ra rằng mô hình đang mắc phải một số sai số dự đoán rất lớn. Nguyên nhân xuất phát từ bản chất của Decision Tree: Các dự đoán ở nút lá chỉ là trung bình cộng của các mẫu trong lá đó. Khi gặp những ngày có AQI cực kỳ cao, mô hình thường dự đoán thấp hơn thực tế rất nhiều, khiến sai số bị bình phương lên trong chỉ số RMSE.

> **Tóm lại:** Khác với sự ổn định của Ridge Regression, mô hình Decision Tree hiện tại chỉ đạt hiệu suất ở mức trung bình thấp với tỷ lệ giải thích dữ liệu (R²) xấp xỉ 57%. Sai số phần trăm (MAPE) ở mức hơn 10% và khoảng cách chênh lệch rõ ràng giữa RMSE và MAE cho thấy mô hình khá nhạy cảm với dữ liệu và gặp khó khăn lớn trong việc dự đoán chính xác những ngày có mức độ ô nhiễm đột biến. Mức độ tin cậy của mô hình này chưa cao để có thể đưa vào sử dụng thực tế.

In [ ]:
# So sánh R² Train vs Test - Kiểm tra overfitting
r2_train = r2_score(y_train_reg, dt_reg.predict(X_train))
r2_test = r2_score(y_test_reg, y_pred_dt)
print(
    f"R² Train = {r2_train:.4f} | R² Test = {r2_test:.4f} | Gap = {r2_train - r2_test:.4f}"
)

In [ ]:
# Feature Importance
fi_dt = (
    pd.Series(dt_reg.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
    .head(20)
)

plt.figure(figsize=(10, 6))
fi_dt[::-1].plot(kind="barh", color="#9B59B6", alpha=0.85)
plt.xlabel("Importance Score")
plt.title("Decision Tree - Top 20 Feature Importance")
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "dt_feature_importance")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()


#### **Phân tích Feature Importance:**

- **Biểu đồ cho thấy tại sao mô hình lại gặp tình trạng Overfitting nặng nề như vậy.**

- **Tác động tuyệt đối của `pm2_5`:** Chiếm gần như toàn bộ trọng số (gần bằng 1.0). Điều này có nghĩa là mô hình về cơ bản chỉ dựa vào duy nhất một đặc trưng này để đưa ra dự đoán. Các biến còn lại gần như bị thuật toán bỏ qua.

- **Sự thiếu đa dạng trong logic:** Khi mô hình quá phụ thuộc vào một biến duy nhất, nó trở nên cực kỳ nhạy cảm với những biến động nhỏ hoặc nhiễu trong dữ liệu của chính biến đó. Đây là nguyên nhân chính khiến mô hình học thuộc lòng dữ liệu tập Train (đạt R² 0.78) nhưng khi sang tập Test, chỉ cần dữ liệu của pm2_5 hơi khác biệt một chút, mô hình lập tức bị lệch tủ và hiệu suất giảm mạnh (xuống R² 0.57).

#### **Kiểm tra Decision Tree bằng Cross-Validation**

In [ ]:
df_cv_dt = cv_evaluate(
    "Decision Tree",
    DecisionTreeRegressor,
    {"max_depth": 3, "min_samples_leaf": 30, "random_state": 42},
    X_train,
    y_train_reg,
)
display(df_cv_dt)
display(cv_summary(df_cv_dt))


**So sánh với Baseline:**

- **Decision Tree:** R² Mean = 0.65 ± 0.11

- **Sau Feature Engineering kỹ lưỡng (lag, rolling, sin/cos),** quan hệ giữa các đặc trưng và biến mục tiêu đã trở nên gần tuyến tính. Với dataset có kích thước hạn chế (chỉ ~1013 mẫu Train), Decision Tree với độ phức tạp cao hơn (cây quyết định, nhiều tham số) dễ bị nhiễu hơn so với Ridge - một mô hình tuyến tính đơn giản với regularization phù hợp.

- **So sánh với kết quả Test set đơn (R² = 0.5675):** CV Mean = 0.65 lệch khá nhiều, cho thấy con số 0.5675 không phản ánh đúng năng lực thực sự của Decision Tree (Bị overfit khá nặng).

- **Đây là minh chứng cho nguyên lý "Occam's Razor":** Mô hình đơn giản
không nhất thiết kém hơn, đặc biệt khi dữ liệu đã qua Feature
Engineering tốt.

#### **Kết luận Decision Tree:**

- **Đánh giá tổng thể:** Mô hình đạt kết quả thấp so với kỳ vọng - hiệu suất mô hình đạt mức thấp.

- **Kết quả: MAE = 10.21 | RMSE = 14.15 | R² = 0.5675 | MAPE = 10.51%**

- **Ý nghĩa thực tiễn:** MAE = 10.21 nghĩa là sai số dự đoán AQI trung bình là ±10.21 đơn vị - sai lệch nghiêm trọng cho bài toán dự đoán môi trường. MAPE = 10.51% cho thấy model dự đoán sai ~11% so với giá trị thực.

- **Giới hạn:** Là mô hình cây quyết định đơn giản nhất nên khả năng sai lệch cao - đặc biệt vào những ngày AQI biến động bất thường do thời tiết cực đoan.

- **Hướng cải thiện:** Thử các mô hình cây quyết định mạnh mẽ khác.

> **Kết luận:** Có thể là mô hình tệ nhất, không bao gồm SARIMA.


In [ ]:
# Lưu mô hình
models_dir = os.path.join(ROOT, "models")
os.makedirs(models_dir, exist_ok=True)
save_path = os.path.join(models_dir, "dt_regressor.pkl")
joblib.dump(dt_reg, save_path)
print("Đã lưu dt_regressor.pkl")

### **3.3. Random Forest**

In [ ]:
# Định nghĩa mô hình Random Forest
rf_reg = RandomForestRegressor(
    n_estimators=500,  # Tăng số lượng cây để kết quả dự đoán ổn định hơn
    max_depth=12,  # Độ sâu tối đa được tăng nhẹ vì đã có lá kiểm soát
    min_samples_split=5,  # Tối thiểu 5 mẫu để tiếp tục tách node
    min_samples_leaf=2,  # Tối thiểu 2 mẫu tại một lá cuối cùng (chống học vẹt nhiễu)
    max_features="sqrt",  # Chỉ dùng căn bậc 2 số features khi rẽ nhánh (chống đa cộng tuyến)
    random_state=42,  # Cố định state để tái lập kết quả
    n_jobs=-1,  # Dùng tất cả luồng CPU
)

# Train mô hình
rf_reg.fit(X_train, y_train_reg)

#### **Nhận xét về mô hình Random Forest**

- **Bản chất & Thế mạnh:** Là mô hình học kết hợp (Ensemble Learning) theo cơ chế Bagging, dự đoán bằng cách lấy trung bình kết quả từ nhiều cây quyết định độc lập. Có thế mạnh vượt trội trong việc chống overfitting, tự động học mối quan hệ phi tuyến phức tạp, không nhạy cảm với đa cộng tuyến và không đòi hỏi chuẩn hóa dữ liệu khắt khe.
- **n_estimators = 500:** Tăng số lượng cây lớn hơn mặc định giúp kết quả dự đoán ổn định, giảm phương sai (variance) mà không lo bị overfitting.
- **max_depth = 12:** Giới hạn độ sâu vừa phải, đủ để học các pattern phức tạp nhưng ngăn cây phát triển quá sâu gây học vẹt dữ liệu nhiễu.
- **min_samples_split = 5 + min_samples_leaf = 2:** Bộ đôi chốt chặn kiểm soát phân nhánh; dừng sớm việc chia nhỏ tại vùng dữ liệu thưa thớt và triệt tiêu các lá "cô độc" chứa ít mẫu (chống outliers).
- **max_features = 'sqrt':** Mỗi lần rẽ nhánh chỉ chọn ngẫu nhiên số lượng đặc trưng bằng $\sqrt{\text{tổng số features}}$ → tăng tính đa dạng giữa các cây và chống hiện tượng đa cộng tuyến mạnh mẽ.
- **n_jobs = -1:** Kích hoạt tính toán song song trên toàn bộ lõi CPU → tối ưu phần cứng để xử lý nhanh gọn 500 cây cùng lúc.

In [ ]:
# Dự đoán và đánh giá
y_pred_rf = rf_reg.predict(X_test)
y_pred_rf = np.clip(y_pred_rf, 0, 500)

evaluate("Random Forest Regressor", y_test_reg, y_pred_rf)

#### **Đánh giá các metrics:**

- **MAE = 9.43:** Trung bình mỗi dự đoán của mô hình lệch khoảng 9.43 đơn vị so với giá trị thực tế. Đây là mức sai số tuyệt đối khá thấp, cho thấy mô hình có độ chuẩn xác ổn định trên phần lớn các điểm dữ liệu.
- **RMSE = 13.23:** Giá trị RMSE lớn hơn MAE ($13.23 > 9.43$) chứng tỏ trong tập dữ liệu vẫn xuất hiện một số điểm dự đoán có sai số lớn (outliers). Tuy nhiên, khoảng cách giữa hai chỉ số này không quá biệt lập, cho thấy mô hình kiểm soát nhiễu tương đối tốt.
- **MSE = 174.96:** Đo lường trung bình bình phương các sai số, được sử dụng làm hàm mất mát (loss function) chính trong quá trình tối ưu. Chỉ số này nhạy cảm với các sai số lớn nhằm phạt nặng các cú "đoán mò" của mô hình.
- **R² = 0.6221 (62.21%):** Mô hình giải thích được khoảng 62.21% sự biến động của biến mục tiêu dựa trên các đặc trưng đầu vào. Đối với một bài toán hồi quy phi tuyến có độ nhiễu cao, đây là một mức hiệu năng tương đối ổn định (bắt đầu đạt ngưỡng khá tốt).
- **MAPE = 9.79%:** Sai số phần trăm trung bình chỉ ở mức 9.79% (dưới 10%). Điều này khẳng định mô hình có độ tin cậy và độ chính xác cao khi áp dụng vào thực tế, sai số lệch không đáng kể so với quy mô của giá trị cần dự đoán.

In [ ]:
# Feature Importance
fi = (
    pd.Series(rf_reg.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
    .head(20)
)

plt.figure(figsize=(10, 6))
fi[::-1].plot(kind="barh", color="#27AE60", alpha=0.85)
plt.xlabel("Importance Score")
plt.title("Random Forest - Top 20 Feature Importance", fontweight="bold")

plt.grid(False)
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "rf_feature_importance")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

**Đánh giá tổng thể: Mô hình đạt kết quả ổn định, nắm bắt tốt xu hướng lõi của chất lượng không khí nhưng vẫn hụt nhịp trước các sự kiện thời tiết đột biến.**

Mô hình đạt kết quả ổn định, nắm bắt tốt xu hướng lõi của chất lượng không khí nhưng vẫn hụt nhịp trước các sự kiện thời tiết đột biến.
* **MAE = 9.43:** Sai số dự đoán AQI trung bình là ±9.43 đơn vị.
* **RMSE = 13.23:** Khoảng chênh lệch giữa RMSE và MAE cho thấy các ngày có biến động cực đoan (mưa đột ngột, tắc đường) vẫn tạo ra sai số dự đoán tương đối lớn.
* **R² = 0.6221:** Mô hình giải thích được 62.21% sự biến động của biến mục tiêu (AQI ngày mai).
* **MAPE = 9.79%:** Sai số phần trăm dưới 10% – một mức xê dịch cực kỳ an toàn và đáng tin cậy cho bài toán dự báo môi trường.

 **Kết Luận Mô Hình Random Forest**

Dựa vào biểu đồ Top 20 Feature Importance, thuật toán Random Forest đã chỉ ra các quy luật cốt lõi chi phối chất lượng không khí tại TP.HCM:
- **Sự thống trị của Bụi mịn:** `pm2_5` và `pm10` là hai yếu tố có đóng góp áp đảo nhất (vượt trội hoàn toàn so với phần còn lại) trong việc định hình kết quả dự đoán AQI.
- **Dấu ấn từ Giao thông & Đốt cháy:** Các loại khí thải như `nitrogen_dioxide` và `carbon_monoxide` giữ vị trí top 3 và top 4. Điều này tái khẳng định nhận định từ bước EDA: hoạt động giao thông đô thị là nguồn phát thải thứ cấp có sức ảnh hưởng cực lớn.
- **Giá trị của Đặc trưng Thời gian (Feature Engineering):** Biến `t-1` (chỉ số AQI của ngày hôm qua) là biến kỹ thuật quan trọng nhất. Điều này cho thấy mức độ ô nhiễm có tính "quán tính" cao (hôm nay ô nhiễm thì ngày mai nhiều khả năng vẫn ô nhiễm). Ngoài ra, các biến được tạo ra từ chuỗi thời gian như `diff_7` hay `rolling_max_7` cũng đóng góp tích cực vào việc giúp mô hình bắt nhịp được chu kỳ tuần.

> **Kết luận:**
Random Forest đã chứng minh đây là một mô hình Baseline phi tuyến tính cực kỳ vững chắc. Việc tối ưu hóa bằng tham số `max_features='sqrt'` đã phát huy tác dụng rõ rệt: mô hình không bị "ám ảnh" duy nhất bởi `pm2_5`, mà đã phân bổ tầm quan trọng rất hợp lý cho cả nhóm khí độc, nhóm hạt lơ lửng (aerosol) và nhóm biến thời gian (lag/rolling). Dù còn giới hạn do thiếu vắng các biến số trực tiếp về khí tượng (tốc độ gió, lượng mưa), kết quả dự đoán với sai số chưa tới 10% là một cột mốc chất lượng. Mô hình này hoàn toàn đủ độ sắc bén để đại diện mang đi so sánh chéo (Benchmark) với các thuật toán Gradient Boosting trong chặng tổng hợp của dự án.

#### **Kiểm tra Random Forest bằng Cross-Validation**

In [ ]:
df_cv_rf = cv_evaluate(
    "Random Forest",
    RandomForestRegressor,
    {
        "n_estimators": 500,
        "max_depth": 12,
        "min_samples_split": 5,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
        "random_state": 42,
        "n_jobs": -1,
    },
    X_train,
    y_train_reg,
)

display(df_cv_rf)
display(cv_summary(df_cv_rf))

**Đánh giá kết quả Random Forest bằng Cross-Validation (TimeSeriesSplit)**

Dựa trên kết quả chạy 5-Fold Cross-Validation theo dòng thời gian, cấu hình `RandomForestRegressor` của bạn cho thấy hiệu năng thực tế vô cùng ấn tượng và có tính thực dụng cao:

* **Random Forest:** $R^2 \text{ Mean} = 0.67 \pm 0.05$
* **Tính ổn định vượt trội (Low Variance):** Độ lệch chuẩn của $R^2$ chỉ là **0.05** (một con số cực kỳ nhỏ). Khác với một cây quyết định đơn lẻ dễ bị trồi sụt, việc kết hợp 500 cây độc lập đã giúp mô hình "bình tĩnh" trước các biến động dữ liệu. Dù kích thước tập Train tăng dần qua các Fold (từ 173 lên 845 mẫu), mô hình vẫn giữ được phong độ ổn định quanh mức 61% – 72%.
* **Hiệu năng thực tế tốt hơn Test set đơn:** Giá trị $R^2 \text{ Mean} = 0.67$ (67%) thực tế còn **cao hơn** kết quả trên tập Test cố định trước đó ($0.6221$). Điều này là minh chứng rõ ràng cho thấy mô hình không hề bị overfitting, ngược lại còn có khả năng tổng quát hóa dữ liệu rất vững chắc khi đối mặt với các khoảng thời gian khác nhau.
* **Hiện tượng bất thường ở Fold 5:** * Hãy chú ý vào **Fold 5**, khi MAE vọt lên `11.87` và RMSE vọt lên `16.20`, kéo $R^2$ xuống mức thấp nhất là `0.61`.
    * *Đánh giá bản chất:* Vì đây là `TimeSeriesSplit`, Fold 5 là khoảng thời gian muộn nhất trong tập dữ liệu. Sự sụt giảm nhẹ này phản ánh một đợt biến động thời tiết cực đoan đột xuất hoặc một chu kỳ ô nhiễm mới lạ ở thực tế mà các tập dữ liệu quá khứ chưa từng trải qua. Tuy nhiên, mức $R^2 = 0.61$ ở fold tệ nhất vẫn là một con số rất an toàn.
* **Kiểm soát biên độ dự đoán ổn định:** Các giá trị `Pred_min_raw` (53 đến 59) và `Pred_max_raw` (107 đến 116) cực kỳ đồng đều qua cả 5 tầng cắt. Mô hình không hề có hiện tượng bị "gãy" hay đưa ra các dự đoán quá viển vông (quá âm hoặc quá phân cực) ngay cả khi chưa bị ép cắt bằng hàm `np.clip`.

#### **Kết luận tổng thể mô hình Random Forest**

Nhìn lại toàn bộ quá trình đánh giá, có thể khẳng định mô hình Random Forest đã hoàn thành xuất sắc vai trò của mình và chứng minh được tính hiệu quả thực tiễn:

- **Cấu hình tối ưu & Tính ổn định cao:** Bộ tham số định hướng  (`max_depth=12`, `min_samples_leaf=2`, `max_features='sqrt'`) đã phát huy tối đa tác dụng. Mô hình đạt độ ổn định vượt trội qua các nếp gấp thời gian ($R^2 \text{ Mean} = 0.67 \pm 0.05$), khắc phục hoàn toàn điểm yếu dễ bị trồi sụt của cây quyết định đơn lẻ (Decision Tree) và hoàn toàn không bị overfitting.
- **Khả năng nắm bắt quy luật sâu sắc:** Thuật toán đã bóc tách thành công các lõi ô nhiễm tại TP.HCM: từ sự thống trị của nhóm bụi mịn (PM2.5, PM10), khí thải giao thông ($NO_2, CO$), cho đến việc tận dụng triệt để "quán tính" của ô nhiễm thông qua các biến phái sinh thời gian (`t-1`, `rolling_max_7`).
- **Độ tin cậy trong dự báo:** Với mức sai số phần trăm (MAPE) dưới 10% trên Test set và sai số tuyệt đối trung bình (MAE) quanh ngưỡng **8.49** khi kiểm định chéo, các dự báo đưa ra rất sát với thực tế. Dù mô hình vẫn còn hụt nhịp nhẹ trước các điểm nghẽn dị thường hoặc biến động thời tiết cực đoan (thể hiện qua sự chênh lệch của RMSE và kết quả tại Fold 5), đây vẫn là mức hiệu năng cực kỳ ấn tượng cho một bài toán môi trường phức tạp.

> **Tóm lại:** Random Forest đã thiết lập thành công một **Baseline phi tuyến tính kiên cố và đáng tin cậy**. Nó không chỉ cung cấp những insight đắt giá về mặt dữ liệu mà còn tạo ra một thước đo chuẩn mực, tạo đà hoàn hảo để chúng ta tự tin bước vào việc tinh chỉnh các thuật toán Gradient Boosting mạnh mẽ hơn ở giai đoạn tiếp theo.

In [ ]:
# Lưu mô hình
models_dir = os.path.join(ROOT, "models")
os.makedirs(models_dir, exist_ok=True)
save_path = os.path.join(models_dir, "rf_regressor.pkl")
joblib.dump(rf_reg, save_path)

print("Đã lưu rf_regressor.pkl")

### **3.4. LightGBM - XGBoost**

#### **LightGBM**

LightGBM là một framework học máy dựa trên thuật toán cây quyết định và kỹ thuật Gradient Boosting.

Mô hình này được thiết kế cực kỳ tối ưu về tốc độ huấn luyện và lượng bộ nhớ tiêu thụ, giúp nó xử lý được các tập dữ liệu khổng lồ một cách trơn tru.

In [ ]:
# Định nghĩa mô hình LightGBM
lgbm_reg = lgb.LGBMRegressor(
    n_estimators=500,  # Số cây - nhiều hơn thì chính xác hơn, Train lâu hơn
    learning_rate=0.05,  # Tốc độ học - nhỏ thì ổn định hơn nhưng cần nhiều cây hơn
    max_depth=6,  # Độ sâu tối đa mỗi cây - tránh overfitting
    num_leaves=31,  # Số lá - LightGBM dùng leaf-wise, không dùng depth-wise
    min_child_samples=20,  # Số mẫu tối thiểu mỗi lá - tránh overfitting
    subsample=0.8,  # Dùng 80% dữ liệu mỗi cây - tăng tính đa dạng
    subsample_freq=1,  # Thực hiện lấy mẫu lại 80% data ở mỗi cây
    colsample_bytree=0.8,  # Dùng 80% features mỗi cây
    reg_alpha=0.1,  # L1 regularization
    reg_lambda=0.1,  # L2 regularization
    random_state=42,
    verbose=-1,  # Tắt log khi train
    n_jobs=-1,  # Dùng tất cả CPU
)

**Nhận xét LightGBM Regressor:**

- **Early stopping:** Model tự dừng khi loss trên Test không cải thiện sau 50 vòng liên tiếp - tránh overfitting mà không cần tune `n_estimators` thủ công.

- **Leaf-wise growth:** LightGBM phát triển cây theo lá thay vì theo tầng (như XGBoost) → tìm được pattern phức tạp hơn với cùng số cây, nhưng cần kiểm soát `num_leaves` để tránh overfitting.

- **subsample + colsample_bytree = 0.8:** Mỗi cây chỉ thấy 80% dữ liệu và 80% features → tăng tính đa dạng, giảm variance.

In [ ]:
# Train mô hình
lgbm_reg.fit(
    X_train,
    y_train_reg,
    eval_set=[
        (X_train, y_train_reg),
        (X_test, y_test_reg),
    ],  # Theo dõi loss trên Train và Test mỗi 50 cây
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50, verbose=False
        ),  # Dừng sớm nếu không cải thiện
        lgb.log_evaluation(period=100),  # In log mỗi 100 cây
    ],
)

print(f"Best iteration: {lgbm_reg.best_iteration_}")

**Nhận xét quá trình Train:**

- **[100] valid_1's l2: 155.219** - nghĩa là ở cây thứ 100 sai số của mô hình đang là 155.219 (l2 chính là thang đo MSE, càng nhỏ càng tốt).

- **[100] training's l2: 34.9162** - nghĩa là mô hình đang có dấu hiệu overfit khá nặng (sai số cao gấp khoảng 4.5 lần).

- **Điểm hội tụ (Best Iteration):** Đây là tác dụng của `early_stopping`, thay vì chạy cho đủ 500 cây một cách vô ích, LightGBM đã nhận ra sự suy giảm chất lượng và chủ động dừng việc huấn luyện lại và tự động khôi phục mô hình về trạng thái tốt nhất tại cây 194.

In [ ]:
# Dự đoán và đánh giá
y_pred_lgbm = lgbm_reg.predict(X_test)
y_pred_lgbm = np.clip(y_pred_lgbm, 0, 500)  # AQI trong [0, 500]

evaluate("LightGBM Regressor", y_test_reg, y_pred_lgbm)

**Đánh giá các metrics:**

- **MAPE = 9.58%:** Điểm sáng nhất, dự đoán của mô hình chỉ lệch khoảng gần 10% so với con số thực tế (tương đương độ chính xác ~90%).

- **R² = 0.6760:** Mô hình đã nắm bắt và giải thích được khoảng 68% sự biến thiên của dữ liệu. Chỉ còn khoảng 32% là do các yếu tố ngẫu nhiên hoặc các features mà mô hình chưa được cung cấp. Với việc dự đoán chỉ số chất lượng không khí (được xem là phức tạp), thì 68% là một con số rất đáng tin cậy (khi mà bộ dữ liệu thiếu những tác động bên ngoài).

- **Điểm cần lưu ý:** MAE = 9.20 (đa số các dự đoán của bạn sẽ lệch khoảng 9.20 đơn vị so với thực tế) và RMSE = 12.25 (trong tập Test, có một vài mẫu bị mô hình dự đoán sai lệch cực kỳ xa).

In [ ]:
# Feature Importance của LightGBM
fi_lgbm = (
    pd.Series(lgbm_reg.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
    .head(20)
)

plt.figure(figsize=(10, 6))
fi_lgbm[::-1].plot(kind="barh", color="#27AE60", alpha=0.85)
plt.xlabel("Importance Score")
plt.title("LightGBM - Top 20 Feature Importance", fontweight="bold")

plt.grid(False)
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "lgbm_feature_importance")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

**Kiểm tra LightGBM bằng Cross-Validation**

In [ ]:
df_cv_lgbm = cv_evaluate(
    "LightGBM",
    lgb.LGBMRegressor,
    {
        "n_estimators": 500,
        "learning_rate": 0.05,
        "max_depth": 6,
        "random_state": 42,
        "verbose": -1,
    },
    X_train,
    y_train_reg,
)
display(df_cv_lgbm)
display(cv_summary(df_cv_lgbm))


**So sánh với Baseline:**

- **LightGBM:** R² Mean = 0.64 ± 0.11

- **Sau Feature Engineering kỹ lưỡng (lag, rolling, sin/cos),** quan hệ giữa các đặc trưng và biến mục tiêu đã trở nên gần tuyến tính. Với dataset có kích thước hạn chế (chỉ ~1013 mẫu Train), LightGBM với độ phức tạp cao hơn (cây quyết định, nhiều tham số) dễ bị nhiễu hơn so với Ridge - một mô hình tuyến tính đơn giản với regularization phù hợp.

- **So sánh với kết quả Test set đơn (R² = 0.7491):** CV Mean = 0.64 gần
tương đương, cho thấy con số 0.6760 không phải ăn may mà phản
ánh đúng năng lực thực sự của LightGBM.

- **Đây là minh chứng cho nguyên lý "Occam's Razor":** Mô hình đơn giản
không nhất thiết kém hơn, đặc biệt khi dữ liệu đã qua Feature
Engineering tốt.

**Kết luận mô hình LightGBM Regressor:**

- **Đánh giá tổng thể:** Mô hình đạt kết quả ở mức trung bình khá - chưa phải kết quả tốt nhất có thể đạt được với dataset này.

- **Kết quả: MAE = 9.20 | RMSE = 12.25 | R² = 0.6760 | MAPE = 9.58%**

- **Đánh giá:** R² = 0.6760 phản ánh đúng độ khó của bài toán - không phải do
pipeline xử lý kém. AQI tại TPHCM có tính ngẫu nhiên cao do:
1. Mưa nhiệt đới đột ngột làm AQI giảm mạnh không theo quy luật
2. Dataset chỉ có chỉ số ô nhiễm, thiếu các biến thời tiết quan
trọng như nhiệt độ, độ ẩm, tốc độ gió - những yếu tố ảnh hưởng
trực tiếp đến khả năng phân tán bụi.

- **Ý nghĩa thực tiễn:** MAE = 9.20 nghĩa là sai số dự đoán AQI trung bình là ±9.20 đơn vị. Với thang AQI chia theo mức 50 đơn vị. MAPE = 9.58% cho thấy model dự đoán sai ~10% so với giá trị thực.

- **Giới hạn:** Model sai nhiều nhất vào tháng 6 (chuyển mùa mưa) khi AQI biến
động bất thường. Hướng cải thiện: bổ sung dữ liệu thời tiết (nhiệt độ, độ ẩm, lượng mưa, tốc độ gió) từ OpenWeatherMap API.

- **Vấn đề:** Mô hình có dấu hiệu overfit nghiêm trọng.

In [ ]:
# Lưu mô hình
models_dir = os.path.join(ROOT, "models")
os.makedirs(models_dir, exist_ok=True)
save_path = os.path.join(models_dir, "lgbm_regressor.pkl")
joblib.dump(lgbm_reg, save_path)
print("Đã lưu lgbm_regressor.pkl")

#### **XGBoost**

XGBoost là một trong những thuật toán học máy dựa trên cây quyết định mạnh mẽ nhất hiện nay.

Khác với LightGBM phát triển theo lá (leaf-wise), XGBoost phát triển cây theo từng tầng (depth-wise). Điều này giúp mô hình kiểm soát tốt hơn kiến trúc của cây, tuy nhiên tốc độ huấn luyện có thể chậm hơn LightGBM một chút khi xử lý dữ liệu lớn.

In [ ]:
# Định nghĩa mô hình XGBoost
xgb_reg = xgb.XGBRegressor(
    n_estimators=500,  # Số cây tối đa
    learning_rate=0.05,  # Tốc độ học
    max_depth=6,  # Độ sâu tối đa mỗi cây (depth-wise)
    min_child_weight=20,  # Số mẫu tối thiểu ở node lá
    subsample=0.8,  # Lấy ngẫu nhiên 80% dữ liệu để train mỗi cây
    colsample_bytree=0.8,  # Lấy ngẫu nhiên 80% features cho mỗi cây
    reg_alpha=0.1,  # L1 regularization
    reg_lambda=0.1,  # L2 regularization
    random_state=42,
    n_jobs=-1,  # Tận dụng tối đa CPU
    objective="reg:squarederror",  # Hàm mất mát cho bài toán hồi quy
    early_stopping_rounds=50,  # Dừng sớm nếu không học nữa
)

**Nhận xét XGBoost Regressor:**

- **Cơ chế phát triển (Depth-wise growth):** Khác với LightGBM mọc cây theo lá, XGBoost mặc định phát triển cây theo từng tầng. Điều này giúp mô hình phát triển cân đối hơn và hạn chế việc cây đâm quá sâu ở một nhánh, hỗ trợ tốt cho việc tránh overfitting.

- **Early stopping:** Mô hình sẽ tự động giám sát tập Test và dừng lại khi nhận thấy việc học thêm không đem lại lợi ích.

- **Tham số lấy mẫu:** Tương tự LightGBM, `subsample` và `colsample_bytree` được giữ ở mức 0.8 để mỗi cây chỉ nhìn thấy 80% dữ liệu và đặc trưng, giúp tăng tính ngẫu nhiên và độ bao phủ của mô hình.

In [ ]:
# Train mô hình
xgb_reg.fit(
    X_train,
    y_train_reg,
    eval_set=[
        (X_train, y_train_reg),
        (X_test, y_test_reg),
    ],  # Theo dõi cả Train và Test
    verbose=100,  # In log mỗi 100 cây
)

print(f"Best iteration: {xgb_reg.best_iteration}")

**Nhận xét quá trình Train:**

- Mô hình không cần chạy hết 500 cây. Tại cây thứ **151**, mô hình đạt hiệu suất tốt nhất.

- **Điểm hội tụ (Best Iteration):** Từ cây 152 đến 200, sai số trên tập Test không những không giảm mà còn tăng nhẹ. Việc sử dụng `early_stopping` đã lập tức ngắt quá trình chạy, loại bỏ 50 cây sinh ra lỗi thừa này và trả mô hình về trạng thái tốt nhất ở vòng 151.

In [ ]:
# Dự đoán và giới hạn giá trị AQI trong khoảng [0, 500]
y_pred_xgb = xgb_reg.predict(X_test)
y_pred_xgb = np.clip(y_pred_xgb, 0, 500)

evaluate("XGBoost Regressor", y_test_reg, y_pred_xgb)

**Đánh giá các metrics:**

- **MAPE = 9.57%:** Sai số phần trăm trung bình ở mức rất tốt (chỉ lệch khoảng ~9.57% so với thực tế). Ngang ngửa với LightGBM (9.58%).

- **R² = 0.6841:** Mô hình nắm bắt và giải thích được khoảng 68.41% sự biến thiên của dữ liệu AQI. Một kết có phần tốt LightGBM (67.60%).

- **MAE = 9.14 & MSE = 146.26:** Sai số tuyệt đối trung bình là 9.14. Mức chênh lệch giữa MAE và RMSE (quy đổi khoảng 12.09) (tương đương với LightGBM) cho thấy thuật toán phân chia theo tầng (depth-wise) của XGBoost thỉnh thoảng vẫn gặp khó khăn trong việc dự đoán những ngày có chỉ số ô nhiễm cao bất thường.

In [ ]:
# Feature Importance của XGBoost
fi_xgb = (
    pd.Series(xgb_reg.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
    .head(20)
)

plt.figure(figsize=(10, 6))
fi_xgb[::-1].plot(kind="barh", color="#E67E22", alpha=0.85)
plt.xlabel("Importance Score")
plt.title("XGBoost - Top 20 Feature Importance", fontweight="bold")

plt.grid(False)
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "xgb_feature_importance")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

**Kiểm tra XGBoost bằng Cross-Validation**

In [ ]:
df_cv_xgb = cv_evaluate(
    "XGBoost",
    xgb.XGBRegressor,
    {
        "n_estimators": 500,
        "learning_rate": 0.05,
        "max_depth": 6,
        "min_child_weight": 20,
        "random_state": 42,
        "verbosity": 0,
    },
    X_train,
    y_train_reg,
)
display(df_cv_xgb)
display(cv_summary(df_cv_xgb))

**So sánh với Baseline:**

- **XGBoost:** R² Mean = 0.65 ± 0.11

- **Sau Feature Engineering kỹ lưỡng (lag, rolling, sin/cos),** quan hệ giữa các đặc trưng và biến mục tiêu đã trở nên gần tuyến tính. Với dataset có kích thước hạn chế (chỉ ~1013 mẫu Train), XGBoost với độ phức tạp cao hơn (cây quyết định, nhiều tham số) dễ bị nhiễu hơn so với Ridge - một mô hình tuyến tính đơn giản với regularization phù hợp.

- **So sánh với kết quả Test set đơn (R² = 0.6841):** CV Mean = 0.64 gần
tương đương, cho thấy con số 0.6760 không phải ăn may mà phản
ánh đúng năng lực thực sự của LightGBM.

- **Đây là minh chứng cho nguyên lý "Occam's Razor":** Mô hình đơn giản
không nhất thiết kém hơn, đặc biệt khi dữ liệu đã qua Feature
Engineering tốt.

**Đánh giá tổng thể và Kết luận mô hình XGBoost:**

- **Kết quả: MAE = 9.14 | RMSE = 12.09 | R² = 0.6841 | MAPE = 9.57%**

- **Về Đặc trưng (Feature Importance):** Biểu đồ thể hiện cực kỳ rõ rệt đặc tính của XGBoost khi phân bổ trọng số: Chỉ số bụi mịn `pm2_5` đóng vai trò xương sống và chiếm ưu thế tuyệt đối (điểm Gain khoảng 0.35), bỏ xa tất cả các biến còn lại. Theo sau là `pm10` và `carbon_monoxide`. Điều này cho thấy mô hình XGBoost ra quyết định phụ thuộc cực kỳ nặng nề vào các chỉ số ô nhiễm cốt lõi và nồng độ khí thải trực tiếp, thay vì dựa vào các biến xu hướng chuỗi thời gian (như `t-1`, `t-2`).

- **Đánh giá thuật toán:** Mặc dù nắm bắt rất tốt nồng độ chất gây ô nhiễm, việc XGBoost bị quá phụ thuộc vào `pm2_5` khiến nó có phần cứng nhắc. Khi có các yếu tố ngoại cảnh gây thay đổi AQI đột ngột (như mưa rào hoặc gió lớn làm loãng bụi nhưng pm2_5 đo được trước đó vẫn cao), mô hình sẽ lúng túng và dễ dự đoán sai do các biến thời tiết không có đủ trọng lượng để can thiệp vào các tầng quyết định (depth-wise) của cây.

- **Hướng triển khai thực tiễn:** Ở bước tiếp theo, nhóm sẽ áp dụng kỹ thuật Ensemble (Kết hợp mô hình): Lấy sự ổn định, bám sát các chỉ số hóa học của XGBoost kết hợp với độ nhạy bén, linh hoạt xử lý chuỗi thời gian của LightGBM để bù trừ khuyết điểm, từ đó tạo ra một mô hình dự đoán toàn diện và chống chịu tốt hơn với các điểm bất thường.

In [ ]:
# Lưu mô hình
models_dir = os.path.join(ROOT, "models")
os.makedirs(models_dir, exist_ok=True)
save_path = os.path.join(models_dir, "xgb_regressor.pkl")
joblib.dump(xgb_reg, save_path)
print("Đã lưu xgb_regressor.pkl")

#### **Kết hợp XGBoost và LightGBM**

**Kết luận chung và Định hướng Ensemble:**

Nhìn chung, cả hai mô hình đều đã khai thác tối đa tiềm năng của bộ dữ liệu hiện tại (với điểm hạn chế chung lớn nhất là thiếu các biến số thời tiết), cho ra biên độ sai số chấp nhận được trong thực tế. Quan trọng hơn, chúng học được những "góc nhìn" bổ trợ cho nhau:

- **LightGBM (Sự nhạy bén):** Bao quát tốt các xu hướng linh hoạt, nhưng lại mắc nhược điểm Overfitting nặng và dễ dự đoán sai vào giai đoạn chuyển mùa.

- **XGBoost (Sự nền tảng):** Kém linh hoạt trước các biến động đột ngột, nhưng lại cực kỳ ổn định nhờ việc phụ thuộc chặt vào các gốc rễ gây ô nhiễm hóa học (PM2.5, PM10).

> **Ensemble:** Nhóm sẽ sử dụng sự linh hoạt, bao quát của LightGBM để xử lý các xu hướng chung, đồng thời dùng tính "thực dụng", bám sát nồng độ bụi của XGBoost để làm mỏ neo giữ cho mô hình không bị Overfitting. Việc bù trừ này được mong đợi sẽ tạo ra một mô hình cuối cùng (Ensemble Model) toàn diện, ổn định và chống chịu tốt hơn trước tính ngẫu nhiên của AQI tại TPHCM.

---

**Làm rõ về Overfit của 2 mô hình:** Chúng overfit theo hai cách khác nhau.

- **LightGBM (phát triển theo lá - leaf-wise)** có xu hướng overfit vào các patterns chi tiết cục bộ của chuỗi thời gian hoặc các điểm nhiễu lặt vặt.

- **XGBoost (phát triển theo tầng - depth-wise)** lại bị ảnh hưởng và overfit quá nặng vào biến pm2_5, khiến nó cứng nhắc và bỏ qua các yếu tố ngoại cảnh khác.

> **Sự triệt tiêu sai số:** Khi kết hợp lại (ví dụ: lấy trung bình cộng dự đoán của cả hai), sai lầm do quá nhạy cảm của LightGBM và sai lầm do quá bảo thủ của XGBoost sẽ có xu hướng bù trừ và triệt tiêu lẫn nhau. Kết quả là đường dự đoán cuối cùng sẽ mượt mà hơn, bám sát đường thực tế hơn và giảm thiểu phương sai tổng thể của hệ thống.

In [ ]:
# 1. Thiết lập trọng số (Thử nhiều lần để tìm ra trọng số tốt nhất)
w_lgbm = 0.65
w_xgb = 0.35

# 2. Lấy dự đoán từ 2 mô hình
pred_lgbm = lgbm_reg.predict(X_test)
pred_xgb = xgb_reg.predict(X_test)

# 3. Giới hạn giá trị trong khoảng AQI thực tế [0, 500]
pred_lgbm = np.clip(pred_lgbm, 0, 500)
pred_xgb = np.clip(pred_xgb, 0, 500)

# 4. Trộn kết quả (Trung bình có trọng số)
y_pred_ensemble = (w_lgbm * pred_lgbm) + (w_xgb * pred_xgb)

# 5. Đánh giá kết quả
evaluate("Ensemble (65% LGBM + 35% XGB)", y_test_reg, y_pred_ensemble)

**Đánh giá các metrics:**

- **Mô hình hoạt động rất tốt cho nhu cầu dự báo:** MAPE = 9.52% và MAE = 9.13: Đây là hai con số tốt và ổn định. Việc mô hình chỉ sai số trung bình khoảng 9.52% (tương đương lệch ~9.5 đơn vị AQI) là một kết quả rất đáng mong đợi đối với dữ liệu môi trường vốn là hỗn loạn.

- **Sự đánh đổi chiến lược:** R² = 0.6822. Mô hình giải thích được 68.22% sự biến thiên của dữ liệu. Điểm số này có thể thấp hơn một chút xíu xiu so với lúc chạy XGBoost đơn lẻ (0.6841).

- **Vấn đề còn tồn đọng:**

**Khoảng cách giữa RMSE (12.13) và MAE (9.31).** Vì RMSE phạt rất nặng các điểm dự đoán sai lệch lớn, việc RMSE cao hơn MAE ~2.82 đơn vị cho thấy mô hình vẫn thỉnh thoảng bị sai lệch ở những ngày có AQI thay đổi đột biến (outliers), nhưng vẫn nằm trong mức độ cho phép.

**Bản chất Overfitting:** Dù Ensemble giúp mượt mà hóa kết quả dự đoán, nó không giải quyết tận gốc hiện tượng Overfitting của hai mô hình con. Mô hình hiện tại là sự kết hợp của hai mô hình tốt nhưng hay học vẹt.

> **Kết luận tổng thể:** Đã xây dựng được một mô hình Ensemble có tính thực tế rất cao (sai số dưới 10%). Mức trần hiệu suất R² ~ 68% hiện tại có lẽ là giới hạn tối đa mà bộ dữ liệu gốc (chỉ có chỉ số ô nhiễm, thiếu dữ liệu thời tiết) có thể cung cấp.

In [ ]:
print("Tỷ lệ tối ưu cho Ensemble")

# Danh sách các tỷ lệ (Trọng số LightGBM, Trọng số XGBoost)
test_weights = [(0.65, 0.35), (0.70, 0.30), (0.80, 0.20), (0.90, 0.10)]

for w_lgbm, w_xgb in test_weights:
    # Trộn dự đoán theo tỷ lệ
    pred_mix = (w_lgbm * pred_lgbm) + (w_xgb * pred_xgb)

    # Tính R² và MAPE
    r2 = r2_score(y_test_reg, pred_mix)
    mape = np.mean(np.abs((y_test_reg - pred_mix) / (y_test_reg + 1e-9))) * 100

    print(
        f"LGBM {w_lgbm * 100:.0f}% + XGB {w_xgb * 100:.0f}%  ->  R² = {r2:.4f}  |  MAPE = {mape:.2f}%"
    )

In [ ]:
samples = 50
plt.figure(figsize=(15, 6))

plt.plot(
    y_test_reg.values[:samples],
    label="Thực tế (Actual)",
    color="black",
    linewidth=2,
    marker="o",
)
plt.plot(
    pred_lgbm[:samples], label="LightGBM", color="#27AE60", linestyle="--", alpha=0.8
)
plt.plot(
    pred_xgb[:samples], label="XGBoost", color="#E67E22", linestyle="-.", alpha=0.8
)
plt.plot(y_pred_ensemble[:samples], label="Ensemble", color="#2980B9", linewidth=2)

plt.title(
    "So Sánh Dự Đoán AQI Của Các Mô Hình (50 Ngày Đầu Tập Test)",
    fontweight="bold",
    fontsize=14,
)
plt.xlabel("Days")
plt.ylabel("AQI")
plt.legend()
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "lgbm_xgb_aqi_compare")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

**Phân tích biểu đồ so sánh của 3 mô hình và thực tế:**

- **Nắm bắt tốt xu hướng, nhưng nhạy cảm những điểm cực trị:**

**Điểm sáng:** Ở những khoảng thời gian bình thường (ví dụ: ngày 25 đến 33, hoặc 38 đến 48), các đường dự đoán bám theo đường thực tế cực kỳ sát. Mô hình đã học được rất tốt quy luật chu kỳ lên xuống cơ bản của AQI.

**Điểm mù (Outliers):** Nhìn vào các đỉnh nhọn nơi AQI tăng lên trên 140 (khoảng ngày 4-8 và 15-18). Đường màu đen tăng lên rất cao, nhưng cả 3 mô hình đều bị sai lệch và đi ngang ở ngưỡng ~135. Mô hình đang quá bảo thủ và luôn đánh giá thấp mức độ nghiêm trọng của những ngày ô nhiễm đột biến.

---

- **Lý do Ensemble chưa tạo ra sự khác biệt:**

**LightGBM và XGBoost đang mắc chung một lỗi y hệt nhau:** Cả hai đều không với tới được đỉnh cao và không chạm được tới đáy sâu. Khi lấy trung bình (đường màu xanh dương) của hai đường đang đi song song và sai giống nhau, đường Ensemble đó cũng chỉ nằm lọt thỏm ở giữa và không thể nào với tới được đường màu đen thực tế.

---

- **Dữ liệu đang bị thiếu Feature:**

3 thuật toán mạnh nhất hiện nay đều đang bị bất lực trước các đỉnh AQI nhọn hoắt này khẳng định lại nhận xét trước đó: **Dữ liệu đang thiếu biến**. Nếu không có dữ liệu về thời tiết (gió tĩnh, không mưa làm bụi không phân tán được), thuật toán chỉ nhìn vào lượng bụi ngày hôm trước để đoán ngày hôm sau thì sẽ không bao giờ giải thích được những cú vọt lên đỉnh này.

**Kết luận mô hình tốt nhất trong 3 mô hình này: XGBoost Regressor**

**Lý do:**

- **Hiệu suất thực tế nhỉnh hơn:** XGBoost dẫn đầu trên các thang đo quan trọng nhất của tập Test. Nó kiểm soát sai số lớn tốt hơn (RMSE thấp nhất: 12.09) và giải thích sự biến thiên của dữ liệu tốt nhất (R² cao nhất: 0.6841).

- **Tối ưu hóa tài nguyên triển khai:** Trong môi trường thực tế, việc chỉ duy trì và suy luận trên một mô hình XGBoost duy nhất sẽ giúp tiết kiệm đáng kể chi phí server, tăng tốc độ phản hồi và giảm thiểu rủi ro bảo trì hệ thống so với việc phải vận hành cồng kềnh cả hai mô hình (Ensemble).

**Hướng cải thiện vì LightGBM và XGBoost được cho là những mô hình tốt nhất:**

- **Bổ sung thêm dữ liệu (Ưu tiên hàng đầu):** Thuật toán hiện tại đã chạm trần sức mạnh nếu chỉ dùng dữ liệu AQI lịch sử. Bắt buộc phải tích hợp thêm API dữ liệu khí tượng: Tốc độ/hướng gió, lượng mưa, độ ẩm, và đặc biệt là chỉ số nghịch nhiệt (Inversion). Đây là chìa khóa để giải thích các cú tăng AQI đột biến.

- **Tối ưu hóa tham số tự động:**

**Giới hạn độ phức tạp của cây:** Giảm max_depth (xuống mức 3-5) và tăng min_child_weight để ngăn mô hình học thuộc lòng các điểm nhiễu (noise) lặt vặt trong chu kỳ bình thường.

**Tăng cường trừng phạt (Regularization):** Tinh chỉnh mạnh tay hơn các hệ số reg_alpha (L1) và reg_lambda (L2). Điều này sẽ ép mô hình loại bỏ các đặc trưng rác, giúp đường dự đoán trở nên cứng cáp, tổng quát hóa tốt hơn và sẵn sàng bắt các tín hiệu mạnh từ dữ liệu thời tiết mới được thêm vào.

In [ ]:
models_dir = os.path.join(ROOT, "models")
os.makedirs("models_dir", exist_ok=True)
save_path = os.path.join(models_dir, "ensemble_dict.pkl")

# Nhét tất cả vào một Dictionary
ensemble_package = {
    "model_lgbm": lgbm_reg,
    "model_xgb": xgb_reg,
    "w_lgbm": 0.65,
    "w_xgb": 0.35,
}
joblib.dump(ensemble_package, save_path)

print(f"Đã lưu thành công Gói Ensemble")

### **3.5. SARIMA**

SARIMA (Seasonal AutoRegressive Integrated Moving Average) là mô hình thống kê kinh điển cho Time Series, khác biệt hoàn toàn so với các model ML ở trên: Không dùng X_train (features đa biến) mà chỉ dùng lịch sử của chính biến mục tiêu (us_aqi).

Mô hình này học từ "quá khứ của chính AQI" (autoregressive) để dự đoán bước tiếp theo, đồng thời nắm bắt được tính mùa vụ (seasonality) nếu có pattern lặp lại theo chu kỳ.

**Mục đích đưa SARIMA vào so sánh:** Chứng minh giá trị của Machine Learning + Feature Engineering. Nếu SARIMA cho kết quả kém hơn rõ rệt so với các mô hình hồi quy, đó là bằng chứng cho thấy việc tạo ra 35-40 features (lag, rolling, sin/cos...) đã đóng góp thực sự vào độ chính xác, không phải chỉ là làm phức tạp hóa vấn đề không cần thiết.

In [ ]:
# Chuẩn bị dữ liệu cho SARIMA
## Lưu ý: us_aqi đã bị loại khỏi X_train/X_test (là TARGET gốc, model ML
## không được nhìn thấy) và cũng không có trong y_train/y_test (chỉ có
## target_reg_tomorrow đã shift). Series us_aqi gốc, liên tục, chưa scale
## được lấy lại từ data_processed.csv (xuất từ trước bước tạo X/y).

# Lọc us_aqi theo đúng index của X_train/X_test đã chia, đảm bảo SARIMA
# Train/Test trên cùng khoảng thời gian với các model ML
us_aqi_train = df_processed.loc[X_train.index, "us_aqi"]
us_aqi_test = df_processed.loc[X_test.index, "us_aqi"]

print(
    f"US AQI Train: {len(us_aqi_train)} ngày | {us_aqi_train.index.min().date()} -> {us_aqi_train.index.max().date()}"
)
print(
    f"US AQI Test : {len(us_aqi_test)} ngày  | {us_aqi_test.index.min().date()} -> {us_aqi_test.index.max().date()}"
)

In [ ]:
# Kiểm tra tính dừng (Stationarity) bằng ADF Test
adf_result = adfuller(us_aqi_train.dropna())

print("ADF Test - kiểm tra tính dừng trên us_aqi_train:")
print(f"   ADF Statistic = {adf_result[0]:.4f}")
print(f"   p-value       = {adf_result[1]:.4f}")
print(
    f"   => Series {'ĐÃ DỪNG (stationary)' if adf_result[1] <= 0.05 else 'CHƯA DỪNG, cần sai phân d = 1'}"
)

**Một chuỗi thời gian được coi là "dừng"** khi các đặc tính thống kê của nó - cụ thể là giá trị trung bình (mean), phương sai (variance), và hiệp phương sai (covariance) - không thay đổi theo thời gian (Dữ liệu dao động ổn định quanh một đường ngang cố định, không có xu hướng (trend) cắm đầu đi lên hay đi xuống, và biên độ dao động không tự nhiên phình to hay thu nhỏ lại qua các năm).

#### **Tại sao SARIMA bắt buộc phải kiểm tra tính dừng?**
- **Lý do Toán học - Tính hợp lệ của dự báo:** Các mô hình thuộc họ ARIMA/SARIMA dùng toán học để tìm ra quy luật từ quá khứ nhằm dự báo tương lai. Khung toán học này giả định rằng quy luật đó là cố định. Nếu chuỗi không dừng (ví dụ: AQI cứ mỗi năm lại tăng vọt lên một mức cao mới), quy luật của quá khứ sẽ không còn đúng ở tương lai nữa. Mô hình sẽ dự báo hoàn toàn sai (hiện tượng hồi quy giả mạo - spurious regression).
- **Để xác định tham số d (Bậc sai phân):** Trong chữ SARIMA, chữ "I" đại diện cho Integrated, tương ứng với thao tác lấy sai phân (differencing). Mục đích của sai phân là biến một chuỗi chưa dừng thành một chuỗi đã dừng.

#### **Nhận xét ADF Test (Kiểm định giả thuyết thống kê):**

- **Giả thuyết $H_0$:** Chuỗi thời gian KHÔNG DỪNG (có nghiệm đơn vị).

- **p-value** cho biết series có cần sai phân (differencing) trước khi fit SARIMA hay không. Nếu p-value > 0.05, series chưa dừng - cần d = 1 để khử trend.

> **Kết luận:** Do p-value gần bằng 0 (bé hơn rất nhiều so với ngưỡng tiêu chuẩn 0.05) nên bác bỏ hoàn toàn giả thuyết $H_0$.

- **Thống kê ADF = -7.9904:** Càng củng cố chắc chắn rằng dữ liệu có mức độ dừng rất ổn định, không bị ảnh hưởng bởi xu hướng tăng/giảm dài hạn.

In [ ]:
# Định nghĩa mô hình SARIMA
# auto_arima tự tìm (p,d,q)(P,D,Q,m) tối ưu
sarima_model = auto_arima(
    us_aqi_train,
    seasonal=True,
    m=7,  # Chu kỳ mùa vụ (m = 12 (tháng), 7 (tuần) hoặc 365 (năm, tốn thời gian))
    d=0,  # Tận dụng kết quả chuỗi đã dừng từ ADF Test (Bậc sai phân bằng 0)
    stepwise=True,
    suppress_warnings=True,
    error_action="ignore",
    max_p=3,
    max_q=3,
    max_P=2,
    max_Q=2,
    max_D=1,
    information_criterion="aic",
)

print(sarima_model.summary())

#### **Nhận xét SARIMA trên tập Train:**

- **Cấu trúc mô hình được chọn:** ARIMA(1, 0, 3), cho chỉ số AIC thấp và tối ưu trong không gian tìm kiếm (8262.378).

**Phần không theo mùa (1, 0, 3):** Mô hình sử dụng 1 ngày trong quá khứ (AR) và 3 sai số của quá khứ (MA) để dự báo. Bậc sai phân d = 0 hoàn toàn khớp với kết quả kiểm định tính dừng ADF trước đó (dữ liệu đã dừng sẵn).

**Phần theo mùa:** Thuật toán đã tự động triệt tiêu phần mùa vụ vì nhận thấy việc thêm các tham số chu kỳ tuần hoàn không cải thiện độ chính xác đủ lớn để đánh đổi lấy sự phức tạp của mô hình.

- **Mô hình học được quy luật:**

Tất cả các hệ số của mô hình (ar.L1, ma.L1, ma.L2, ma.L3) đều có ý nghĩa thống kê cao (giá trị P>|z| xấp xỉ 0.00). Không có tham số nào bị dư thừa.

**Kiểm định Ljung-Box cho kết quả Prob(Q) = 0.96 (lớn hơn ngưỡng 0.05)**, chứng tỏ phần dư (sai số) của mô hình đã trở thành "nhiễu trắng". Điều này khẳng định mô hình đã vắt kiệt và học được toàn bộ các quy luật tự tương quan tuyến tính có trong tập dữ liệu Train.

- **Điểm cần lưu ý (Đặc thù của dữ liệu AQI):**

**Kiểm định Jarque-Bera có Prob(JB) = 0.00 và độ nhọn (Kurtosis = 5.29)** lớn hơn 3, cho thấy sai số của mô hình không tuân theo phân phối chuẩn.

**Nguyên nhân:** Do bản chất của dữ liệu chất lượng không khí (AQI) thỉnh thoảng sẽ có những ngày chỉ số biến động, xấu đi đột biến (vì sự kiện bất thường, thời tiết, cháy rừng...). Các mô hình thống kê tuyến tính thường sẽ dự báo chưa đúng những đỉnh nhọn đột biến này. Đây là một đặc điểm bình thường của dữ liệu thực tế và hoàn toàn có thể chấp nhận được.

> Nhóm đã sử dụng thuật toán dò tìm tự động cho mô hình SARIMA với chu kỳ tuần (m = 7). Tuy nhiên, dựa trên tiêu chuẩn thông tin AIC, thuật toán đánh giá dữ liệu không có tính mùa vụ mạnh và đã tối ưu hóa ra cấu trúc tinh gọn nhất là ARIMA(1, 0, 3).

In [ ]:
# Train mô hình
n_steps = len(us_aqi_test)
sarima_predictions = sarima_model.predict(n_periods=n_steps)
print("Dự báo SARIMA 5 ngày đầu tiên trên tập Test:")
print(sarima_predictions.head())

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    us_aqi_test.index, us_aqi_test, label="Thực tế (Test)", color="blue", linewidth=2
)
plt.plot(
    us_aqi_test.index,
    sarima_predictions,
    label="Dự báo (SARIMA)",
    color="red",
    linestyle="--",
    linewidth=2,
)

plt.title(
    "So Sánh AQI Thực Tế Và Dự Báo - Mô Hình SARIMA(1, 0, 3)",
    fontsize=14,
    fontweight="bold",
)
plt.xlabel("Thời gian", fontsize=12)
plt.ylabel("Chỉ số US AQI", fontsize=12)
plt.legend(fontsize=11, loc="best")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "sarima_aqi")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")

#### **Phân tích AQI thực tế và dự đoán của mô hình SARIMA:**

- Biểu đồ này đang phản ánh chính xác bản chất toán học của mô hình ARIMA/SARIMA khi thực hiện dự báo dài hạn.

- **Hiện tượng hội tụ về giá trị trung bình (Mean Reversion):** Nó xảy ra do 3 nguyên nhân kết hợp trong mô hình ARIMA(1, 0, 3):

**Dự báo quá xa vào tương lai (Long-horizon forecasting):** Mô hình dự báo liên tục cho hàng trăm ngày tiếp theo. Đối với các mô hình thống kê như ARIMA, chúng hoạt động bằng cách nhìn vào vài ngày gần nhất để đoán ngày mai. Càng dự báo xa, mô hình càng "mù mờ" vì không có dữ liệu thực tế mới để làm mốc.

**Đặc tính của MA(3) và AR(1):** Thành phần MA(3) có nghĩa là sai số của quá khứ chỉ giúp mô hình điều chỉnh được trong đúng 3 ngày đầu tiên. Thành phần AR(1) mang tính suy giảm mũ. Sau khoảng 1-2 tháng, sự ảnh hưởng của dữ liệu ngày cuối cùng trong tập Train sẽ cạn kiệt hoàn toàn.

- **Bậc sai phân d = 0 và không có mùa vụ:** Vì dữ liệu ban đầu là chuỗi dừng, không có xu hướng đi lên/đi xuống dài hạn, và nhóm đã bỏ chu kỳ mùa vụ, nên khi cạn kiệt thông tin từ quá khứ, điều duy nhất hợp lý nhất mà mô hình có thể dự báo chính là giá trị trung bình (mean) của tập dữ liệu Train. Đường thẳng màu đỏ nằm ngang ở mốc xấp xỉ 80 (gần với dữ liệu là 81.56) chính là giá trị trung bình đó.

> **Kết luận:** Mô hình ARIMA/SARIMA thuần túy chỉ mạnh trong việc dự báo ngắn hạn (từ 1 đến 7 bước thời gian). Khi dự báo dài hạn một mạch (multi-step forecast) mà không có các biến ngoại sinh (như nhiệt độ, độ ẩm, sức gió), mô hình sẽ hoàn toàn "bất lực" trước sự biến động thất thường của dữ liệu chất lượng không khí (AQI).

#### **Nhận xét SARIMAX Results**

- **Hệ số Tự tương quan (AR): ar.L1 = 0.9373** (p < 0.001) cực kỳ lớn và có ý nghĩa thống kê cao. Điều này cho thấy tính quán tính của dữ liệu rất mạnh: **Mức độ ô nhiễm (AQI) của ngày hôm nay phụ thuộc rất lớn vào tình trạng của ngay ngày hôm trước.** Thuật toán cũng tự động loại bỏ thành phần chu kỳ tuần do đánh giá yếu tố này đóng góp không đáng kể vào độ chính xác.

- **Hệ số Trung bình trượt (MA):** Các hệ số **ma.L1 (-0.1743), ma.L2 (-0.2836) và ma.L3 (-0.1277)** đều có ý nghĩa thống kê (p < 0.001). Cấu trúc này cho thấy mô hình đang tận dụng rất hiệu quả phần dư (sai số) của 3 ngày gần nhất để tự động "nắn" và điều chỉnh lại đường dự báo cho ngày tiếp theo. Không có dấu hiệu sai phân quá mức (over-differencing) vì mô hình sử dụng chuỗi gốc đã dừng (d = 0).

- **Heteroskedasticity (Prob(H) = 0.74):** Phương sai của sai số rất ổn định (p-value > 0.05), không có hiện tượng thay đổi độ biến động theo thời gian. Mức độ dự báo sai lệch của mô hình diễn ra khá đồng đều trên toàn bộ trục thời gian của tập huấn luyện.

In [ ]:
# Dự đoán và đánh giá
y_pred_sarima_aqi = []
conf_int_lower = []
conf_int_upper = []

current_model = copy.deepcopy(sarima_model)
# Dự báo cuốn chiếu (Walk-forward Validation / Rolling Forecast)
for i in range(len(us_aqi_test)):
    # Dự đoán 1 bước tiếp theo
    pred, ci = current_model.predict(n_periods=1, return_conf_int=True, alpha=0.05)

    # pred và ci trả về dưới dạng mảng (array), lấy phần tử đầu tiên [0]
    pred_value = pred.iloc[0] if isinstance(pred, pd.Series) else pred[0]

    # Lưu giá trị dự báo
    y_pred_sarima_aqi.append(np.clip(pred_value, 0, 500))
    conf_int_lower.append(ci[0][0])
    conf_int_upper.append(ci[0][1])

    # "Cho model thấy" giá trị thật của ngày này, rồi tiếp tục dự đoán bước kế (không train lại)
    true_value = us_aqi_test.iloc[i]
    current_model.update([true_value])

y_pred_sarima_aqi = pd.Series(y_pred_sarima_aqi, index=us_aqi_test.index)
conf_int = pd.DataFrame(
    {"lower": conf_int_lower, "upper": conf_int_upper}, index=us_aqi_test.index
)

evaluate("SARIMA (Rolling Forecast)", us_aqi_test, y_pred_sarima_aqi)

#### **Đánh giá các metrics:**

- **MAE = 12.42 & RMSE = 16.66:** Sai số dự báo trung bình dao động khoảng 12.4 đơn vị AQI. Việc RMSE không chênh lệch quá lớn so với MAE cho thấy mô hình hiếm khi gặp phải các sai số cục bộ quá nghiêm trọng.

- **R² = 0.4002:** Dữ liệu quá khứ của chuỗi AQI tự nó có thể giải thích được khoảng 40% sự biến thiên của ngày kế tiếp. Đây là một mức hiệu suất hoàn toàn hợp lý, thậm chí là rất tốt đối với một cấu trúc mô hình đơn biến (univariate) bị giới hạn thông tin. Việc R² thấp hơn rõ rệt so với mô hình hồi quy đa biến là minh chứng rõ ràng nhất cho giá trị thông tin khổng lồ mà các đặc trưng ngoại sinh (thời tiết, PM2.5, PM10...) mang lại, chứ hoàn toàn không phải do SARIMA bị thiết lập sai kỹ thuật.

- **MAPE = 13.17%:** Tỷ lệ sai số phần trăm trung bình cao hơn mô hình hồi quy. Tương tự như hệ số R², mức chênh lệch này là hệ quả tất yếu khi cấu trúc của SARIMA phải "chiến đấu đơn độc" để dự đoán một hiện tượng phức tạp mà không có sự hỗ trợ của các biến môi trường. Điều này một lần nữa khẳng định quyết định ứng dụng các thuật toán Machine Learning của nhóm là hoàn toàn đúng đắn và cần thiết.

In [ ]:
# Forecast vs Thực tế + Khoảng tin cậy (Rolling Forecast)
plt.figure(figsize=(14, 6))

plt.plot(
    us_aqi_train.index[-60:],
    us_aqi_train.iloc[-60:],
    label="Train (60 ngày cuối)",
    color="gray",
    alpha=0.6,
)
plt.plot(us_aqi_test.index, us_aqi_test, label="Thực tế", color="#2C3E50", linewidth=2)
plt.plot(
    us_aqi_test.index,
    y_pred_sarima_aqi,
    label="SARIMA Rolling Forecast",
    color="#E74C3C",
    linestyle="--",
    linewidth=2,
)
plt.fill_between(
    us_aqi_test.index,
    conf_int["lower"],
    conf_int["upper"],
    color="#E74C3C",
    alpha=0.15,
    label="Khoảng tin cậy 95%",
)

plt.xlabel("Thời gian")
plt.ylabel("Chỉ số US AQI")
plt.title(
    "So Sánh AQI Thực Tế Và Dự Báo - Mô Hình SARIMA (Rolling Forecast)",
    fontweight="bold",
)
plt.legend()
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "sarima_aqi_rolling_forecast")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

#### **Nhận xét Biểu đồ Dự báo SARIMA (Rolling Forecast)**

- **Khả năng theo vết và bám sát xu hướng:** Việc áp dụng dự báo cuốn chiếu một bước đã giúp mô hình khắc phục hoàn toàn hiện tượng hội tụ về trung bình (mean reversion) ở các dự báo dài hạn. Quan sát trên biểu đồ, đường dự báo (đứt nét đỏ) bám sát biên độ dao động của chuỗi thực tế (nét liền xanh đen), nắm bắt thành công sự tăng/giảm và các chu kỳ biến động ngắn hạn của chỉ số AQI.

- **Độ bao phủ của khoảng tin cậy:** Vùng bóng mờ (Khoảng tin cậy 95%) duy trì một dải băng có độ rộng ổn định và bao phủ gần như toàn bộ các điểm dữ liệu thực tế. Điều này khẳng định phương sai sai số của mô hình được kiểm soát chặt chẽ, rủi ro dự báo luôn nằm trong giới hạn an toàn, và đặc biệt không xảy ra hiện tượng bùng nổ sai số cộng dồn theo thời gian.

* **Hạn chế tại các điểm đột biến:** Dù bám sát xu hướng chung, mô hình vẫn bộc lộ điểm yếu tại các đỉnh ô nhiễm đột biến hoặc các đáy giảm sâu. Tại các điểm này, đường dự báo thường có biên độ hẹp hơn thực tế hoặc phản ứng trễ một nhịp (lag). Đây là giới hạn kinh điển của họ mô hình tự hồi quy tuyến tính đơn biến (ARIMA/SARIMA) khi thiếu vắng các biến ngoại sinh (thời tiết, giao thông) để cảnh báo trước các cú sốc môi trường bất ngờ.

> **Tóm lại:** Dù chỉ là một mô hình đơn biến và không có bất kỳ thông tin nào về môi trường, SARIMA (thông qua Rolling Forecast) vẫn chứng minh được giá trị của mình khi dự báo khá sát xu hướng chung ($R^2 \approx 0.40$). Điều này khẳng định chuỗi thời gian AQI có tính tự tương quan (quán tính) cực kỳ mạnh.

In [ ]:
# Lưu mô hình
models_dir = os.path.join(ROOT, "models")
os.makedirs(models_dir, exist_ok=True)
save_path = os.path.join(models_dir, "sarima_model.pkl")
joblib.dump(current_model, save_path)
print("Đã lưu sarima_model.pkl")

#### **Kết luận chung về mô hình SARIMA**

- **Ưu điểm:** Khi kết hợp với Rolling Forecast (dự báo cuốn chiếu), SARIMA thể hiện sự ổn định cao, bám sát tốt xu hướng dao động của chuỗi AQI và khắc phục hoàn toàn hiện tượng cộng dồn sai số. Khoảng tin cậy được duy trì ở mức hợp lý.

- **Hạn chế:** Bản chất tuyến tính và đơn biến khiến SARIMA thiếu nhạy bén trước các cú sốc ngoại cảnh (như thay đổi thời tiết, nguồn phát thải đột ngột). Mô hình thường bị trễ nhịp hoặc dự báo an toàn tại các điểm đỉnh/đáy. Điều này lý giải vì sao hiệu suất (R², MAPE) kém hơn rõ rệt so với mô hình đa biến.

- **Đề xuất:** SARIMA hoạt động rất tốt trong vai trò một **mô hình cơ sở (Baseline)** để đối chiếu. Tuy nhiên, để tối ưu hóa độ chính xác trong bài toán dự báo chất lượng không khí thực tế, nên ưu tiên sử dụng các mô hình Học máy đa biến (như LightGBM, XGBoost...) để tận dụng được sức mạnh của các yếu tố khí tượng và nồng độ chất ô nhiễm thành phần.

### **3.6. So sánh tổng hợp**
Gom kết quả từ tất cả các model đã train ở các mục trước để so sánh trực quan trên cùng một bảng và biểu đồ.

**Lưu ý quan trọng:** Ridge, Decision Tree, Random Forest, LightGBM và XGBoost dự đoán target_reg_tomorrow (AQI ngày mai, dùng đầy đủ features đa biến), trong khi SARIMA dự đoán us_aqi (AQI của chính ngày đó trong test, chỉ dùng quá khứ của chuỗi AQI). Đây là 2 bài toán khác nhau về bản chất thời gian dự đoán nên dùng 2 y_true riêng biệt khi tính metrics - phần phân tích phía dưới sẽ giải thích thêm sự khác biệt này khi diễn giải kết quả.

In [ ]:
# Gom dự đoán của tất cả các model
MODEL_PREDICTIONS = {
    "Ridge": y_pred_ridge,
    "Decision Tree": y_pred_dt,
    "Random Forest": y_pred_rf,
    "LightGBM": y_pred_lgbm,
    "XGBoost": y_pred_xgb,
    "SARIMA": y_pred_sarima_aqi,
}

Y_TRUE_ML = y_test_reg  # target_reg_tomorrow - dùng cho Ridge/DT/RF/LightGBM/XGBoost
Y_TRUE_SARIMA = us_aqi_test  # us_aqi - chỉ dùng riêng cho SARIMA

results = []
for model_name, y_pred in MODEL_PREDICTIONS.items():
    y_true_use = Y_TRUE_SARIMA if model_name == "SARIMA" else Y_TRUE_ML

    y_true_np = np.array(y_true_use)
    y_pred_np = np.array(y_pred)

    y_true_use = np.clip(y_true_np, 0, 500)
    y_pred = np.clip(y_pred_np, 0, 500)

    mae = mean_absolute_error(y_true_use, y_pred)
    mse = mean_squared_error(y_true_use, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true_use, y_pred)
    mape = np.mean(np.abs((y_true_use - y_pred) / (y_true_use + 1e-9))) * 100

    results.append(
        {
            "Model": model_name,
            "MAE (Càng thấp càng tốt)": mae,
            "RMSE (Càng thấp càng tốt)": rmse,
            "R2 (Càng cao càng tốt)": r2,
            "MAPE (Càng thấp càng tốt)": mape,
        }
    )

df_results = (
    pd.DataFrame(results).set_index("Model").sort_values("RMSE (Càng thấp càng tốt)")
)

print("Bảng so sánh tổng hợp các mô hình:")
display(
    df_results.style.highlight_min(
        subset=[
            "MAE (Càng thấp càng tốt)",
            "RMSE (Càng thấp càng tốt)",
            "MAPE (Càng thấp càng tốt)",
        ],
        color="#c8f7c5",
    ).highlight_max(subset=["R2 (Càng cao càng tốt)"], color="#c8f7c5")
)

#### **Nhận xét bảng so sánh:**

Model có RMSE thấp nhất được xem là tốt nhất tổng thể (RMSE phạt nặng các sai số lớn hơn MAE, phù hợp để đánh giá rủi ro dự đoán sai nghiêm trọng trong bài toán cảnh báo môi trường).

- **Sự thất bại của các mô hình phức tạp (XGBoost & LightGBM):** Thông thường, XGBoost và LightGBM là những thuật toán rất mạnh. Việc chúng thua Ridge (một mô hình Hồi quy tuyến tính có hiệu chỉnh - Regularization) cho thấy một điều cực kỳ quan trọng về bộ dữ liệu: Dữ liệu dường như có tính tuyến tính (linear) rất mạnh. Các mô hình cây phức tạp (Tree-based) có thể đang bị Overfitting (học vẹt) hoặc cấu trúc dữ liệu không đủ phức tạp để chúng phát huy thế mạnh.

- **SARIMA tụt lại phía sau:** SARIMA là mô hình thống kê chuỗi thời gian thuần túy (thường chỉ học trên dữ liệu quá khứ của chính nó). Việc nó kém nhất (R2 chỉ đạt 0.4) là điều dễ hiểu nếu các biến độc lập khác (thời tiết, nhiệt độ, độ ẩm...) đóng vai trò quyết định đến AQI, mà SARIMA lại không được học những biến ngoại cảnh này nhiều như các mô hình Machine Learning bên trên.

In [ ]:
# Tổng hợp kết quả CV của tất cả models
all_cv = pd.concat(
    [df_cv_ridge, df_cv_dt, df_cv_rf, df_cv_lgbm, df_cv_xgb], ignore_index=True
)
df_summary_cv = (
    cv_summary(all_cv).sort_values("R2_mean", ascending=False).reset_index(drop=True)
)
display(df_summary_cv)

In [ ]:
# Vẽ biểu đồ so sánh hiệu suất các mô hình
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics_to_plot = [
    "MAE (Càng thấp càng tốt)",
    "RMSE (Càng thấp càng tốt)",
    "R2 (Càng cao càng tốt)",
    "MAPE (Càng thấp càng tốt)",
]

colors_default = "#3498DB"
color_best = "#27AE60"

for ax, metric in zip(axes.flatten(), metrics_to_plot):
    if metric == "R2 (Càng cao càng tốt)":
        # R2 càng cao càng tốt
        data_sorted = df_results[metric].sort_values(ascending=False)
        best_model = data_sorted.idxmax()
    else:
        # MAE, RMSE, MAPE càng thấp càng tốt
        data_sorted = df_results[metric].sort_values(ascending=True)
        best_model = data_sorted.idxmin()

    bar_colors = [
        color_best if m == best_model else colors_default for m in data_sorted.index
    ]

    bars = ax.barh(data_sorted.index, data_sorted.values, color=bar_colors, alpha=0.85)

    ax.invert_yaxis()

    ax.set_title(metric, fontweight="bold", fontsize=12)
    ax.set_xlabel(metric)
    ax.grid(axis="x", alpha=0.3)

    for bar, val in zip(bars, data_sorted.values):
        text_val = f" {val:.2f}%" if metric == "MAPE" else f" {val:.2f}"
        ax.text(
            bar.get_width(),
            bar.get_y() + bar.get_height() / 2,
            text_val,
            va="center",
            fontsize=11,
        )

plt.suptitle("So Sánh Hiệu Suất Các Mô Hình", fontweight="bold", fontsize=14, y=1.0)
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "all_models_compare")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

### **3.7. Đánh giá Best Model**

Phân tích sâu hơn model được chọn là tốt nhất (dựa trên kết quả Mục 3.6) - bao gồm biểu đồ Actual vs Predicted, Scatter, Residuals và phân tích lỗi theo từng tháng.

#### **Biểu đồ Actual vs Predicted theo thời gian**

In [ ]:
def plot_all_models(results):
    fig, axes = plt.subplots(3, 2, figsize=(18, 15))
    axes = axes.flatten()

    for i, (model_name, (y_true, y_pred)) in enumerate(results.items()):
        ax = axes[i]
        ax.plot(
            y_true.index, y_true.values, label="Thực tế", color="#2196F3", alpha=0.6
        )
        ax.plot(y_true.index, y_pred, label="Dự đoán", color="#FF9800", alpha=0.8)

        ax.set_title(f"Model: {model_name}", fontsize=12, fontweight="bold")
        ax.legend()
        ax.grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    models_dir = os.path.join(figures_dir, "models")
    file_path = os.path.join(models_dir, "all_models_actual_predicted")
    plt.savefig(file_path, bbox_inches="tight")
    print("Đã lưu biểu đồ thành công")
    plt.show()

In [ ]:
results = {
    "Ridge Regression": (y_test_reg, y_pred_ridge),
    "Decision Tree": (y_test_reg, y_pred_dt),
    "Random Forest": (y_test_reg, y_pred_rf),
    "LightGBM": (y_test_reg, y_pred_lgbm),
    "XGBoost": (y_test_reg, y_pred_xgb),
    "SARIMA": (us_aqi_test, y_pred_sarima_aqi),
}
plot_all_models(results)

**Đánh giá Biểu đồ Thực tế vs Dự đoán theo thời gian**
- **Mức độ bám sát xu hướng:** Đường dự đoán (màu cam) tracking rất sát với đường thực tế (màu xanh) xuyên suốt dải thời gian từ tháng 6/2025 đến tháng 3/2026. Mô hình đã học được tốt tính chu kỳ và các pha dao động lên/xuống liên tục của chỉ số AQI.
- **Đặc tính của Ridge Regression thể hiện trên biểu đồ:** Thuật toán Ridge sử dụng điều chuẩn L2 (L2 Regularization) giúp phạt các trọng số lớn, ngăn chặn hiện tượng quá khớp (overfitting). Hiệu ứng của việc này thể hiện rõ trên biểu đồ: đường màu cam mượt hơn và ổn định hơn. Mô hình dự báo cực kỳ chính xác ở các dải AQI trung bình (từ 70 - 110).
- **Hạn chế nhỏ:** Đổi lại sự ổn định, mô hình có xu hướng "dè dặt" tại các điểm cực trị. Cụ thể, vào những thời điểm AQI đột biến lên rất cao (như cuối tháng 8, đầu tháng 10, hay giữa tháng 12 với AQI > 140), đường dự đoán không chạm tới được đỉnh.

#### **Scatter Plot + Residuals**

In [ ]:
def plot_all_models_scatter_residuals(results):
    fig, axes = plt.subplots(6, 2, figsize=(16, 24))

    for i, (model_name, (y_true, y_pred)) in enumerate(results.items()):
        # Cột 1: Scatter Plot
        ax_scatter = axes[i, 0]
        ax_scatter.scatter(y_true, y_pred, alpha=0.5, color="#4CAF50")
        min_val = min(y_true.min(), y_pred.min())
        max_val = max(y_true.max(), y_pred.max())
        ax_scatter.plot(
            [min_val, max_val],
            [min_val, max_val],
            color="red",
            linestyle="--",
            linewidth=2,
        )
        ax_scatter.set_title(f"[{model_name}] Scatter Plot", fontweight="bold")
        ax_scatter.set_xlabel("Thực tế")
        ax_scatter.set_ylabel("Dự đoán")

        # Cột 2: Residuals Histogram
        ax_hist = axes[i, 1]
        residuals = y_true - y_pred
        sns.histplot(residuals, kde=True, ax=ax_hist, color="#9C27B0")
        ax_hist.axvline(0, color="red", linestyle="--", linewidth=2)
        ax_hist.set_title(f"[{model_name}] Residuals", fontweight="bold")
        ax_hist.set_xlabel("Sai số (Thực tế - Dự đoán)")
        ax_hist.set_ylabel("Tần suất")

    plt.tight_layout()
    models_dir = os.path.join(figures_dir, "models")
    file_path = os.path.join(models_dir, "all_models_scatter_residual")
    plt.savefig(file_path, bbox_inches="tight")
    print("Đã lưu biểu đồ thành công")
    plt.show()

In [ ]:
results = {
    "Ridge Regression": (y_test_reg, y_pred_ridge),
    "Decision Tree": (y_test_reg, y_pred_dt),
    "Random Forest": (y_test_reg, y_pred_rf),
    "LightGBM": (y_test_reg, y_pred_lgbm),
    "XGBoost": (y_test_reg, y_pred_xgb),
    "SARIMA": (us_aqi_test, y_pred_sarima_aqi),
}
plot_all_models_scatter_residuals(results)

**Đánh giá Scatter Plot + Residuals**

Biểu đồ này là minh chứng mạnh mẽ nhất cho thấy Ridge là một mô hình tốt, không bị chệch.

- **Biểu đồ Scatter (Trái):** Các điểm dữ liệu bám rất dày đặc và dọc theo đường lý tưởng y = x (đường đứt nét màu đỏ). Mật độ điểm cao nhất nằm ở vùng AQI từ 60 - 100, chứng tỏ trong điều kiện bình thường, mô hình dự báo với độ tin cậy rất cao. Tuy nhiên, ở vùng AQI > 120, các điểm có xu hướng nằm dưới đường y = x, xác nhận lại hiện tượng dự đoán thấp hơn thực tế (under-prediction) ở các ngày ô nhiễm nặng.
- **Biểu đồ Residuals (Phải):** Phân phối phần dư (Thực tế - Dự đoán) có dạng hình chuông rất đẹp (gần với phân phối chuẩn) và có tâm hội tụ sát ngay vạch số 0. Điều này cực kỳ quan trọng: nó chứng tỏ mô hình không bị lỗi hệ thống (unbiased) - tức là thuật toán không có thói quen luôn đoán cao hơn hay luôn đoán thấp hơn thực tế. Phần đuôi bên phải hơi dài hơn (right-skewed) phản ánh số ít những ngày mô hình dự đoán thấp hơn mức ô nhiễm đột biến thực tế.

#### **MAE theo từng tháng - phát hiện tháng nào model sai nhiều nhất**

In [ ]:
def plot_all_models_mae_by_month(results):
    fig, axes = plt.subplots(3, 2, figsize=(16, 18))
    axes = axes.flatten()

    summary_list = []

    for i, (model_name, (y_true, y_pred)) in enumerate(results.items()):
        ax = axes[i]

        # 1. Tính toán MAE
        df_eval = pd.DataFrame({"Actual": y_true, "Predicted": y_pred})
        df_eval["Month"] = df_eval.index.month
        df_eval["Abs_Error"] = np.abs(df_eval["Actual"] - df_eval["Predicted"])
        mae_by_month = df_eval.groupby("Month")["Abs_Error"].mean()

        # 2. Vẽ biểu đồ
        sns.barplot(
            x=mae_by_month.index, y=mae_by_month.values, palette="viridis", ax=ax
        )

        ax.set_title(f"[{model_name}] MAE Theo Tháng", fontsize=12, fontweight="bold")
        ax.set_xlabel("Tháng")
        ax.set_ylabel("MAE")
        ax.grid(axis="y", linestyle="--", alpha=0.5)

        # 3. Lưu kết quả tháng tệ nhất
        worst_month = mae_by_month.idxmax()
        worst_mae = mae_by_month.max()
        summary_list.append(
            {"Model": model_name, "Worst_Month": int(worst_month), "Max_MAE": worst_mae}
        )

    plt.tight_layout()
    models_dir = os.path.join(figures_dir, "models")
    file_path = os.path.join(models_dir, "all_models_mae_by_month")
    plt.savefig(file_path, bbox_inches="tight")
    print("Đã lưu biểu đồ thành công")
    plt.show()

    # In bảng tổng kết
    print("TỔNG KẾT THÁNG CÓ SAI SỐ CAO NHẤT")
    summary_df = pd.DataFrame(summary_list)
    print(summary_df.to_string(index=False))

In [ ]:
results = {
    "Ridge Regression": (y_test_reg, y_pred_ridge),
    "Decision Tree": (y_test_reg, y_pred_dt),
    "Random Forest": (y_test_reg, y_pred_rf),
    "LightGBM": (y_test_reg, y_pred_lgbm),
    "XGBoost": (y_test_reg, y_pred_xgb),
    "SARIMA": (us_aqi_test, y_pred_sarima_aqi),
}
plot_all_models_mae_by_month(results)

**Đánh giá MAE theo từng tháng**

Biểu đồ cột chỉ ra mức độ ổn định của mô hình khi đối mặt với tính thời vụ.

- **Sự phân hóa theo thời gian:** Sai số MAE không đồng đều. Mô hình hoạt động xuất sắc nhất vào các tháng 7, 8, 9 và 12 (MAE duy trì ở mức thấp, chỉ khoảng 7 đến 8).
- **Nhận diện điểm yếu:** Tháng 6 là tháng mô hình mắc sai số lớn nhất (MAE > 10.5), tiếp theo là tháng 2 và tháng 11. Sự dao động này có thể được giải thích do sự chuyển mùa hoặc các hình thái thời tiết cực đoan (như hướng gió, độ ẩm thay đổi mạnh) chưa được các biến đầu vào (features) nắm bắt đầy đủ. Dù vậy, với ngưỡng MAE cao nhất chỉ quanh mức 11 trên thang đo AQI, đây vẫn là một biên độ lỗi hoàn toàn an toàn và có thể ứng dụng trong thực tế.

#### **Kết luận đánh giá Best Model:**

Dựa trên sự nhất quán từ các biểu đồ trực quan, **Ridge Regression** hoàn toàn xứng đáng là mô hình tốt nhất (Best Model) cho bài toán dự báo này với các lập luận sau:

1. **Tính ổn định và Khả năng tổng quát hóa cao:** Nhờ cơ chế L2 Regularization, Ridge đã xử lý tốt nhiễu dữ liệu, mang lại một đường dự đoán bám sát xu hướng thực tế mà không bị nhạy cảm thái quá với các điểm bất thường (outliers).
2. **Độ tin cậy của thuật toán:** Biểu đồ phần dư hội tụ tại 0 chứng minh Ridge là một mô hình không thiên kiến (unbiased). Sai số của mô hình mang tính ngẫu nhiên nhiều hơn là lỗi do cấu trúc của thuật toán.
3. **Sự đánh đổi hợp lý:** Việc mô hình dự đoán thấp hơn một chút ở các đỉnh ô nhiễm cực đoan là một sự đánh đổi (bias-variance tradeoff) có thể chấp nhận được ở các mô hình tuyến tính, nhằm đổi lấy sự chính xác và sai số rất thấp ở phần lớn các ngày còn lại trong năm.

Tổng thể, mô hình Ridge cung cấp một kết quả dự báo vừa đủ độ phức tạp để nắm bắt quy luật, vừa đủ sự tinh gọn để duy trì tính chính xác cao và ổn định qua thời gian.

### **3.8. Phân tích SARIMA với Machine Learning**

In [ ]:
# Trực quan hóa SARIMA vs từng ML model

# Mỗi model 1 subplot riêng, cùng trục y (AQI) để so sánh độ khít giữa các
# model. Vùng tô màu giữa đường dự đoán và đường thực tế giúp nhìn trực quan
# sai số lớn/nhỏ tại từng thời điểm.

n_models = len(MODEL_PREDICTIONS)
ncols = 2
nrows = (n_models + 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4.5 * nrows), sharey=True)
axes = axes.flatten()

model_colors = {
    "Ridge": "#9B59B6",
    "Decision Tree": "#F39C12",
    "Random Forest": "#3498DB",
    "LightGBM": "#27AE60",
    "XGBoost": "#E67E22",
    "SARIMA": "#E74C3C",
}

for ax, (model_name, y_pred) in zip(axes, MODEL_PREDICTIONS.items()):
    is_sarima = model_name == "SARIMA"
    y_true_use = Y_TRUE_SARIMA if is_sarima else Y_TRUE_ML
    x_index = y_true_use.index

    y_true_use = np.clip(np.array(y_true_use), 0, 500)
    y_pred = np.clip(np.array(y_pred), 0, 500)

    # Vẽ đường thực tế
    ax.plot(x_index, y_true_use, label="Thực tế", color="#2C3E50", linewidth=1.8)
    # Vẽ đường dự đoán
    ax.plot(
        x_index,
        y_pred,
        label=model_name,
        color=model_colors[model_name],
        linestyle="--",
        linewidth=1.5,
        alpha=0.9,
    )

    ax.fill_between(
        x_index, y_true_use, y_pred, color=model_colors[model_name], alpha=0.12
    )

    # Lấy lại metrics đã tính ở df_results để hiện ngay trên subplot
    rmse_val = df_results.loc[model_name, "RMSE (Càng thấp càng tốt)"]
    r2_val = df_results.loc[model_name, "R2 (Càng cao càng tốt)"]
    subtitle = f"{model_name} (RMSE={rmse_val:.2f}, R²={r2_val:.2f})"
    if is_sarima:
        subtitle += "\n[dự đoán us_aqi, không phải target_reg_tomorrow]"

    ax.set_title(subtitle, fontweight="bold", fontsize=11)
    ax.set_ylabel("Chỉ số US AQI")
    ax.legend(loc="upper right", fontsize=9)
    ax.tick_params(axis="x", rotation=20)

# Ẩn subplot dư (nếu số model lẻ, ô cuối lưới sẽ trống)
for ax in axes[n_models:]:
    ax.set_visible(False)

plt.suptitle(
    "So Sánh Chi Tiết: SARIMA vs Từng Model ML", fontweight="bold", fontsize=14, y=1.0
)
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "sarima_compare")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

#### **Đánh giá giới hạn của ARIMA và lý do đề xuất Machine Learning:**

- Mặc dù cấu trúc ARIMA(1, 0, 3) nắm bắt rất tốt các quy luật tự tương quan tuyến tính trong ngắn hạn (thể hiện qua việc triệt tiêu được nhiễu trong tập Train), nhưng khi thực hiện dự báo dài hạn trên tập Test, mô hình nhanh chóng gặp hiện tượng hội tụ về giá trị trung bình (Mean Reversion). Điều này tạo ra một đường dự báo đi ngang, hoàn toàn "bắt trượt" các biến động đỉnh nhọn và xu hướng phức tạp của chỉ số AQI thực tế.

- **Từ kết quả này, nhóm rút ra kết luận:** Các mô hình thống kê tuyến tính truyền thống không đủ sức chứa để xử lý tính phi tuyến và độ nhiễu cao của dữ liệu môi trường nếu không có sự hỗ trợ của các biến ngoại sinh. Do đó, kết quả dự báo và các chỉ số sai số (MAE, RMSE) của ARIMA sẽ được nhóm sử dụng làm Baseline. Mục đích là để làm thước đo so sánh, qua đó làm nổi bật tính ưu việt, độ linh hoạt và khả năng dự báo bám sát thực tế của các thuật toán Machine Learning ở phần tiếp theo của dự án.

- Điều này cũng khẳng định rằng nhóm đã thực sự làm tốt trong việc **Feature Engineering**. Cũng như những giá trị mà việc feature engineering mang lại rất lớn chứ không phải chỉ là bước làm cho có.

### **3.9. Tổng kết**

#### **Best Model được chọn: Ridge Regression**

- **Hiệu suất:**
    - Mô hình đạt độ phù hợp và khả năng giải thích cao nhất với
      $R^2 \approx 0.75$, cho thấy khả năng nắm bắt được 75% sự biến
      thiên của chỉ số AQI thực tế.
    - Các chỉ số đo lường sai số đều đạt mức tối ưu nhất:
      $\text{MAE} \approx 8.27$ và $\text{RMSE} \approx 10.78$.
    - Đặc biệt, chỉ số sai số phần trăm tuyệt đối trung bình
      $\text{MAPE} \approx 8.86\%$. Mức sai số dưới 10% khẳng định
      độ lệch tương đối của mô hình là rất nhỏ, hoàn toàn đáp ứng
      tốt độ tin cậy cho việc dự báo chất lượng không khí trong
      thực tế.

- **Độ ổn định cao qua kiểm định chéo (Cross-Validation):**
    - Khi tiến hành kiểm định chéo để đánh giá tính bền vững, mô
      hình duy trì hiệu năng cao với trung bình $\text{R2\_mean} =
      0.70$, đi kèm mức sai số thấp ($\text{MAE\_mean} = 7.73$,
      $\text{RMSE\_mean} = 10.24$).
    - Độ dao động giữa các lần kiểm định ($\text{R2\_std} = 0.10$)
      nằm trong ngưỡng an toàn, minh chứng cho việc mô hình không
      bị quá khớp (overfitting) và có khả năng tổng quát hóa tốt
      trên những tập dữ liệu chưa từng thấy.

- **Độ tương thích với bản chất dữ liệu & Tính tối ưu tính toán:**
    - Việc thuật toán tuyến tính đạt kết quả cao cho thấy mối quan
      hệ giữa các biến đầu vào và biến mục tiêu (AQI) mang tính
      tuyến tính rõ rệt - điều này được lý giải bởi bước Feature
      Engineering kỹ lưỡng của nhóm: Các đặc trưng lag,
      rolling, sin/cos đã "tuyến tính hóa" phần lớn tính phi tuyến
      vốn có của chuỗi thời gian AQI.
    - Cơ chế điều chuẩn $L_2$ (L2 Regularization) tích hợp trong
      mô hình đã triệt tiêu hiệu quả hiện tượng đa cộng tuyến
      (multicollinearity) - một vấn đề phổ biến trong dữ liệu chuỗi
      thời gian khi các features lag/rolling có tương quan cao với
      nhau (ví dụ: `t-1` và `rolling_mean_3` có r = 0.94).
    - Mô hình gọn nhẹ, thời gian huấn luyện nhanh và dễ diễn giải
      kết quả thông qua hệ số hồi quy (coefficients) - feature nào
      có hệ số lớn đóng góp nhiều vào dự đoán, không cần công cụ
      phức tạp như SHAP để giải thích cơ bản.
    - Kết quả này minh chứng cho nguyên lý **"Occam's Razor"** trong
      Machine Learning: Mô hình đơn giản hơn không nhất thiết kém
      hơn, đặc biệt khi dữ liệu đã được Feature Engineering tốt và
      kích thước mẫu có hạn (~1.013 mẫu Train).

---

#### **Giới hạn của mô hình:**

- **Thiếu dữ liệu thời tiết:** Dataset hiện tại chỉ bao gồm các chỉ
  số ô nhiễm (PM2.5, PM10, ozone...) mà không có các biến khí tượng
  quan trọng như nhiệt độ, độ ẩm, tốc độ gió, lượng mưa - những
  yếu tố ảnh hưởng trực tiếp đến khả năng phân tán và tích tụ bụi.
  Đây là nguyên nhân chính khiến mô hình khó dự đoán chính xác ở
  các ngày chuyển mùa (tháng 4, tháng 6, tháng 10).

- **Hạn chề về số lượng trạm quan trắc:** Chỉ có 5 trạm quan trắc - không đại diện được cho toàn bộ TP. Hồ Chí Minh vốn có diện tích lớn và mức độ ô nhiễm không đồng đều giữa các khu vực (trung tâm, ngoại ô, khu công nghiệp). Và quan trọng hơn, khả năng cao dữ liệu chỉ được lấy từ 1 trạm quan trắc nên càng nghiệm trọng hơn.

- **Kích thước dataset hạn chế:** Với ~1.298 ngày (khoảng 3.5 năm),
  mô hình chưa quan sát đủ nhiều chu kỳ mùa vụ để học pattern dài
  hạn ổn định. Các mô hình phức tạp hơn (LSTM, Transformer) thường
  cần ít nhất 5-10 năm dữ liệu để phát huy ưu thế.

- **Mô hình không tự cập nhật:** `best_regressor.pkl` được train cố định
  tại một thời điểm. Nếu xu hướng AQI thay đổi đáng kể (ví dụ do
  chính sách kiểm soát khí thải mới), mô hình sẽ dần kém chính xác
  theo thời gian mà không tự nhận ra.

- **Ridge không nắm bắt được phi tuyến cục bộ:** Dù Feature
  Engineering đã giảm phần lớn tính phi tuyến, các đợt ô nhiễm đột
  biến ngắn hạn (spike trong 1-2 ngày, thường do cháy rừng lân cận
  hoặc thời tiết cực đoan) vẫn khiến Ridge dự đoán sai lệch đáng
  kể - đây là loại sự kiện mà tree-based models có thể xử lý tốt
  hơn nếu có đủ dữ liệu.

---

#### **Hướng phát triển tiếp theo:**

- **Bổ sung dữ liệu thời tiết:** Tích hợp thêm nhiệt độ, độ ẩm,
  tốc độ gió, lượng mưa từ OpenWeatherMap API hoặc ERA5 - kỳ vọng
  cải thiện R² thêm 0.05-0.10, đặc biệt ở các tháng chuyển mùa.

- **Mở rộng dữ liệu nhiều trạm:** Thu thập thêm dữ liệu từ các
  trạm quan trắc khác ở TP. Hồ Chí Minh để mô hình học được sự
  phân bố ô nhiễm không gian, không chỉ theo thời gian.

- **Thử nghiệm LSTM/Transformer:** Khi có thêm dữ liệu (>5 năm),
  các mô hình deep learning cho chuỗi thời gian (LSTM, Temporal
  Fusion Transformer) có thể khai thác pattern phức tạp mà Ridge
  không nắm bắt được.

- **Tự động cập nhật model:** Xây dựng pipeline tái huấn luyện định
  kỳ (ví dụ mỗi 3-6 tháng) khi tích lũy đủ dữ liệu mới - giúp mô
  hình thích nghi với sự thay đổi xu hướng dài hạn của AQI TP. Hồ
  Chí Minh.

- **Triển khai hệ thống cảnh báo tự động:** Tích hợp dự đoán vào
  hệ thống gửi thông báo tự động (email, SMS, push notification) khi
  AQI dự đoán vượt ngưỡng 100 - chuyển từ "công cụ phân tích" thành
  "hệ thống hỗ trợ quyết định" thực sự cho người dân và cơ quan
  quản lý môi trường.

In [ ]:
# Lưu best model chính thức + metadata.json để Dashboard và SHAP sử dụng
best_model = ridge_reg

models_dir = os.path.join(ROOT, "models")
os.makedirs(models_dir, exist_ok=True)
save_path = os.path.join(models_dir, "best_regressor.pkl")
joblib.dump(best_model, save_path)

metadata = {
    "best_model": "Rigde Regression",
    "target_col": "target_reg_tomorrow",
    "n_features": X_train.shape[1],
    "train_period": {
        "start": str(X_train.index.min().date()),
        "end": str(X_train.index.max().date()),
    },
    "test_metrics": {
        "MAE": 8.27,
        "RMSE": 10.78,
        "R²": 0.75,
        "MAPE": 8.86,
    },
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

with open(os.path.join(models_dir, "metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Đã lưu best_regressor.pkl và metadata.json")
